# Rare-disease target prioritisation — CIPA (HSAN IV, *NTRK1*)

**Question.** Given a rare monogenic disease, which genes are plausible starting points for a therapy, and which of them are *understudied* — the dark genome?

**Disease.** Congenital insensitivity to pain with anhidrosis (hereditary sensory and autonomic neuropathy type 4) is caused by biallelic loss-of-function variants in *NTRK1*, which encodes TrkA, the high-affinity NGF receptor. NGF-dependent nociceptors and sympathetic neurons are lost during development. There is no disease-modifying therapy.

**What this notebook does.** A reproducible pipeline over public resources — Open Targets, Pharos/IDG, ClinVar, UniProt, AlphaFold DB (AlphaMissense), GTEx and Tabula Sapiens, and mouse knockout phenotypes — that:

1. collects candidates from three declared sources: disease associations (A), mechanistic neighbours of the causal gene (B), causal genes of sister diseases (C);
2. annotates each on two separate axes, **knowledge** and **chemical matter**, instead of conflating them into "T-dark";
3. keeps automatic steps and **declared curation** apart, with one written reason per manual decision;
4. asks which therapeutic strategy can serve which **genotype**;
5. scores candidates on four axes that are never averaged into one another.

**Scope.** Computational prioritisation only. No experimental validation is performed; proposed experiments are described as future work. To re-run on another disease, change `DISEASE_QUERY` below and revisit the declared-curation cells: the automatic steps transfer, the judgements do not.

**Origin.** Developed from an MSc exam project (*AI applied to Molecular Genomics*, 2025) in which candidates were selected by hand. This notebook re-evaluates that selection systematically.

In [ ]:
# --- Configuration -----------------------------------------------------------------------------------
import json, time, random, datetime, pathlib, re, shutil
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_rows", 250, "display.max_colwidth", 60)

DISEASE_QUERY = "congenital insensitivity to pain with anhidrosis"   # change this to re-run on another disease
OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"
PHAROS_URL = "https://pharos-api.ncats.io/graphql"
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

ROOT = pathlib.Path("/content/rare-target-prioritization")
RAW = ROOT / "data" / "raw"; PROC = ROOT / "data" / "processed"; FIG = ROOT / "figures"
for p in (RAW, PROC, FIG): p.mkdir(parents=True, exist_ok=True)
RUN_DATE = datetime.date.today().isoformat()

# Genes retained by the original exam analysis (Open Targets + Pharos, July 2025).
# Used only to compare 2025 with 2026: the pipeline never reads this list.
EXAM_TDARK = ["MPV17", "NTF3", "PRDM12", "SH2D2A", "SUCLA2", "AWAT2", "FOXE3", "LMX1B", "HAX1"]
EXAM_N_ASSOCIATED = 36
print("Run date:", RUN_DATE, "· working directory:", ROOT)

In [ ]:
# --- Open Targets GraphQL helper -----------------------------------------------------------------------
def ot_query(query, variables=None, retries=3, pause=2.0):
    # On a 4xx the body carries the server's explanation, so it is shown rather than discarded, and the
    # call is not retried: a malformed or too-expensive query stays malformed or too expensive.
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(OT_URL, json={"query": query, "variables": variables or {}}, timeout=60)
            if 400 <= r.status_code < 500 and r.status_code != 429:
                raise RuntimeError(f"HTTP {r.status_code} from Open Targets -- server said:\n{r.text[:2000]}")
            r.raise_for_status()
            payload = r.json()
            if payload.get("errors"):
                raise RuntimeError(json.dumps(payload["errors"], indent=2)[:2000])
            return payload["data"]
        except RuntimeError:
            raise
        except (requests.RequestException, ValueError) as e:
            if attempt == retries:
                raise
            print(f"Attempt {attempt} failed ({e}); retrying in {pause}s")
            time.sleep(pause)

In [ ]:
# --- Optional: restore after a Colab runtime restart ---------------------------------------------------
# Colab clears memory and /content on restart. Rather than repeating every API call, run the two cells
# above and then upload the archive produced by the last cell of a previous session.
import zipfile, io
from google.colab import files
for name, data in files.upload().items():
    zipfile.ZipFile(io.BytesIO(data)).extractall(ROOT)
print("Restored:", sorted(p.name for p in PROC.iterdir()))

## Phase 1 — Disease → associated genes (source A)

The disease is resolved by **name search**, not a hard-coded identifier, so the pipeline transfers unchanged to another disease. Every associated target is retrieved with its evidence broken down by type, and the Open Targets release is recorded in the manifest.

**Outputs:** `01_associated_targets.csv`, `01_manifest.json`, `01_evidence_heatmap.png`.

In [ ]:
# --- Record the Open Targets data release (reproducibility) -------------------
META_Q = "query { meta { name apiVersion { x y z } dataVersion { year month } } }"
try:
    meta = ot_query(META_Q)["meta"]
    print(json.dumps(meta, indent=2))
except Exception as e:
    # Field names can change between API versions; the release is also recoverable from the UI footer.
    meta = {"error": str(e)}
    print("Could not read API metadata -- note the release shown on platform.opentargets.org instead.\n", e)

In [ ]:
# --- 1. Resolve the disease identifier ------------------------------------------
SEARCH_Q = '''
query search($q: String!) {
  search(queryString: $q, entityNames: ["disease"], page: {index: 0, size: 10}) {
    hits { id name entity description }
  }
}'''
hits = ot_query(SEARCH_Q, {"q": DISEASE_QUERY})["search"]["hits"]
hits_df = pd.DataFrame(hits)[["id", "name", "entity"]]
display(hits_df)

exact = [h for h in hits if h["name"].strip().lower() == DISEASE_QUERY.lower()]
DISEASE_ID = (exact or hits)[0]["id"]
DISEASE_NAME = (exact or hits)[0]["name"]
print(f"\nSelected: {DISEASE_ID} -- {DISEASE_NAME}")
print("If this is not the right entry, set DISEASE_ID manually from the table above and re-run from here.")

In [ ]:
# --- 2. All associated targets, with per-datatype evidence ------------------------
ASSOC_Q = '''
query assoc($id: String!, $index: Int!, $size: Int!) {
  disease(efoId: $id) {
    id
    name
    associatedTargets(page: {index: $index, size: $size}) {
      count
      rows {
        score
        datatypeScores { id score }
        target { id approvedSymbol approvedName biotype }
      }
    }
  }
}'''

PAGE = 500
rows, index, total = [], 0, None
while True:
    d = ot_query(ASSOC_Q, {"id": DISEASE_ID, "index": index, "size": PAGE})["disease"]
    block = d["associatedTargets"]
    total = block["count"]
    rows.extend(block["rows"])
    if len(rows) >= total or not block["rows"]:
        break
    index += 1

print(f"{DISEASE_NAME}: {total} associated targets (retrieved {len(rows)})")

In [ ]:
# --- 3. Snapshot: raw JSON + tidy table + manifest -------------------------------
raw_path = RAW / f"ot_associations_{DISEASE_ID}_{RUN_DATE}.json"
raw_path.write_text(json.dumps({"meta": meta, "disease_id": DISEASE_ID, "rows": rows}, indent=1))

records = []
for r in rows:
    rec = {"ensembl_id": r["target"]["id"], "symbol": r["target"]["approvedSymbol"],
           "name": r["target"]["approvedName"], "biotype": r["target"]["biotype"],
           "ot_overall_score": r["score"]}
    rec.update({f"dt_{x['id']}": x["score"] for x in r["datatypeScores"]})
    records.append(rec)

assoc = (pd.DataFrame(records)
           .sort_values("ot_overall_score", ascending=False)
           .reset_index(drop=True))
dt_cols = [c for c in assoc.columns if c.startswith("dt_")]
assoc[dt_cols] = assoc[dt_cols].fillna(0.0)

# Which evidence types support each gene, and whether any is *genetic*
assoc["evidence_types"] = assoc[dt_cols].apply(lambda s: ";".join(c[3:] for c in dt_cols if s[c] > 0), axis=1)
assoc["n_evidence_types"] = (assoc[dt_cols] > 0).sum(axis=1)
assoc["has_genetic_evidence"] = assoc.get("dt_genetic_association", pd.Series(0, index=assoc.index)) > 0
assoc["literature_only"] = assoc["evidence_types"].eq("literature")

csv_path = PROC / "01_associated_targets.csv"
assoc.to_csv(csv_path, index=False)

manifest = {"phase": "01_disease_to_genes", "run_date": RUN_DATE, "disease_query": DISEASE_QUERY,
            "disease_id": DISEASE_ID, "disease_name": DISEASE_NAME, "n_targets": int(total),
            "open_targets_meta": meta, "raw_snapshot": raw_path.name, "table": csv_path.name}
(PROC / "01_manifest.json").write_text(json.dumps(manifest, indent=2))

print("Evidence datatypes present:", [c[3:] for c in dt_cols])
display(assoc.drop(columns=["name"]).head(40))

In [ ]:
# --- 4. Evidence composition -----------------------------------------------------
summary = pd.Series({
    "associated targets": len(assoc),
    "with genetic-association evidence": int(assoc["has_genetic_evidence"].sum()),
    "supported by literature only": int(assoc["literature_only"].sum()),
    "protein-coding": int((assoc["biotype"] == "protein_coding").sum()),
})
display(summary.to_frame("n"))

# Heatmap: which kind of evidence links each gene to the disease (top 40 by overall score)
top = assoc.head(40)
mat = top[dt_cols].to_numpy()
cmap = LinearSegmentedColormap.from_list("ot", ["#F8FAFC", "#34607D", "#0F3D5C"])

fig, ax = plt.subplots(figsize=(1.0 + 0.9 * len(dt_cols), 0.28 * len(top) + 1.5))
im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=0, vmax=1)
ax.set_xticks(range(len(dt_cols)), [c[3:].replace("_", " ") for c in dt_cols], rotation=40, ha="right")
ax.set_yticks(range(len(top)), [f"{s}  ({v:.2f})" for s, v in zip(top["symbol"], top["ot_overall_score"])], fontsize=8)
for i, sym in enumerate(top["symbol"]):
    if sym in EXAM_TDARK:
        ax.get_yticklabels()[i].set_color("#D97706"); ax.get_yticklabels()[i].set_fontweight("bold")
fig.colorbar(im, ax=ax, fraction=0.03, label="datatype score")
ax.set_title(f"{DISEASE_NAME}: evidence behind each association\n(orange = genes retained in the 2025 exam analysis)", fontsize=10)
fig.tight_layout()
fig.savefig(FIG / "01_evidence_heatmap.png", dpi=200)
plt.show()

In [ ]:
# --- 5. Sanity check against the original exam analysis (July 2025) ----------------
print(f"Associated targets now: {len(assoc)}  (exam, July 2025: {EXAM_N_ASSOCIATED})")
check = (pd.DataFrame({"symbol": EXAM_TDARK})
           .merge(assoc[["symbol", "ot_overall_score", "evidence_types", "has_genetic_evidence"]],
                  on="symbol", how="left"))
check["rank"] = check["symbol"].map({s: i + 1 for i, s in enumerate(assoc["symbol"])})
check["still_associated"] = check["ot_overall_score"].notna()
display(check)
# Differences are expected across Open Targets releases; they are reported, not hidden.

## Phase 2 — Candidate universe (sources B and C) and development level

In a monogenic loss-of-function disease an association list contains one cause and a long weak tail, so two further sources are added:

- **B — mechanistic neighbours.** Direct interactors of the causal gene (IntAct ≥ 0.6, SIGNOR, Reactome, STRING ≥ 0.9). Seeds are genes whose genetic-association score reaches the declared threshold.
- **C — sister-disease genes.** Causal genes of diseases sharing a parent class in the ontology. A parent with too many children is treated as too broad to define "sisters".

Each gene keeps the source it came from. Annotation adds tractability, chemical probes and drugs from Open Targets, and the target development level from Pharos/IDG. The 2025 exam criterion — no drugs, no chemical probe, at most three tractability assessments — is also computed explicitly, so the two can be compared rather than assumed equivalent.

In [ ]:
# --- Phase 2 configuration --------------------------------------------------------
# Requires cells 1-2 (imports, ot_query). If the Colab runtime was reset, re-run cells 1-6 first.
if "assoc" not in globals():
    assoc = pd.read_csv(PROC / "01_associated_targets.csv")
    _m1 = json.loads((PROC / "01_manifest.json").read_text())
    DISEASE_ID, DISEASE_NAME = _m1["disease_id"], _m1["disease_name"]
dt_cols = [c for c in assoc.columns if c.startswith("dt_")]

# Declared thresholds (all reported in the manifest)
CAUSAL_GENETIC_MIN = 0.5      # genetic_association score that makes a gene "causal" (seeds of B; members of C)
INTERACTION_SOURCES = {       # source B: database -> minimum interaction score (None = curated, no score)
    "intact": 0.6, "signor": None, "reactome": None, "string": 0.9}
MAX_SIBLINGS = 25             # a parent disease class with more children is too broad to define "sister diseases"
PHAROS_URL = "https://pharos-api.ncats.io/graphql"

gen = assoc.get("dt_genetic_association", pd.Series(0.0, index=assoc.index))
seeds = assoc.loc[gen >= CAUSAL_GENETIC_MIN, ["ensembl_id", "symbol"]]
if seeds.empty:                                  # fallback: top-ranked gene
    seeds = assoc.head(1)[["ensembl_id", "symbol"]]
SEEDS = dict(zip(seeds["ensembl_id"], seeds["symbol"]))
print("Causal seed gene(s):", SEEDS)

In [ ]:
# --- Source B: mechanistic neighbours of the causal gene(s) ------------------------
INTER_Q = '''
query inter($id: String!, $src: InteractionSourceEnum, $thr: Float, $size: Int!) {
  target(ensemblId: $id) {
    interactions(sourceDatabase: $src, scoreThreshold: $thr, page: {index: 0, size: $size}) {
      count
      rows { score sourceDatabase targetB { id approvedSymbol } }
    }
  }
}'''

edges = []
for seed_id, seed_sym in SEEDS.items():
    for src, thr in INTERACTION_SOURCES.items():
        t = ot_query(INTER_Q, {"id": seed_id, "src": src, "thr": thr, "size": 1000})["target"]
        block = (t or {}).get("interactions") or {"count": 0, "rows": []}
        if block["count"] > len(block["rows"]):
            print(f"  warning: {seed_sym}/{src} truncated ({len(block['rows'])}/{block['count']})")
        for r in block["rows"]:
            tb = r.get("targetB")
            if tb and tb["id"] != seed_id:       # human-mapped partners only, no self-loops
                edges.append({"seed": seed_sym, "ensembl_id": tb["id"], "symbol": tb["approvedSymbol"],
                              "source_db": src, "score": r["score"]})
        print(f"{seed_sym} · {src:<8} threshold={thr}: {block['count']} interactions")

edges = pd.DataFrame(edges, columns=["seed", "ensembl_id", "symbol", "source_db", "score"])
B = (edges.groupby(["ensembl_id", "symbol"])
          .agg(b_seed=("seed", lambda s: ";".join(sorted(set(s)))),
               b_sources=("source_db", lambda s: ";".join(sorted(set(s)))),
               b_n_sources=("source_db", "nunique"),
               b_max_score=("score", "max"))
          .reset_index()
          .sort_values(["b_n_sources", "b_max_score"], ascending=False))
print(f"\nSource B: {len(B)} distinct partners")
display(B.head(30))

In [ ]:
# --- Source C: causal genes of sister diseases ----------------------------------------
FAM_Q = '''
query fam($id: String!) {
  disease(efoId: $id) {
    parents { id name children { id name } }
  }
}'''
SIB_Q = '''
query sib($id: String!) {
  disease(efoId: $id) {
    id name
    associatedTargets(page: {index: 0, size: 500}) {
      count
      rows { datatypeScores { id score } target { id approvedSymbol } }
    }
  }
}'''

parents = ot_query(FAM_Q, {"id": DISEASE_ID})["disease"]["parents"]
par_tab = pd.DataFrame([{"parent_id": p["id"], "parent": p["name"], "n_children": len(p["children"]),
                         "used": len(p["children"]) <= MAX_SIBLINGS} for p in parents])
display(par_tab)

siblings = {c["id"]: c["name"] for p in parents if len(p["children"]) <= MAX_SIBLINGS
            for c in p["children"] if c["id"] != DISEASE_ID}
print(f"{len(siblings)} sister diseases:", "; ".join(siblings.values()))

c_rows = []
for sid, sname in siblings.items():
    d = ot_query(SIB_Q, {"id": sid})["disease"]
    for r in d["associatedTargets"]["rows"]:
        g = {x["id"]: x["score"] for x in r["datatypeScores"]}.get("genetic_association", 0.0)
        if g >= CAUSAL_GENETIC_MIN:
            c_rows.append({"ensembl_id": r["target"]["id"], "symbol": r["target"]["approvedSymbol"],
                           "sister_disease": sname, "genetic_score": g})

c_rows = pd.DataFrame(c_rows, columns=["ensembl_id", "symbol", "sister_disease", "genetic_score"])
C = (c_rows.groupby(["ensembl_id", "symbol"])
           .agg(c_diseases=("sister_disease", lambda s: "; ".join(sorted(set(s)))),
                c_max_genetic=("genetic_score", "max"))
           .reset_index()
           .sort_values("c_max_genetic", ascending=False))
print(f"\nSource C: {len(C)} causal genes of sister diseases")
display(C)

In [ ]:
# --- Candidate universe: A ∪ B ∪ C, every gene labelled by origin ----------------------
A = assoc[["ensembl_id", "symbol", "ot_overall_score", "evidence_types", "has_genetic_evidence"]].copy()
universe = (A.merge(B, on=["ensembl_id", "symbol"], how="outer")
             .merge(C, on=["ensembl_id", "symbol"], how="outer"))
universe["src_A"] = universe["ot_overall_score"].notna()
universe["src_B"] = universe["b_sources"].notna()
universe["src_C"] = universe["c_diseases"].notna()
universe["is_seed"] = universe["ensembl_id"].isin(SEEDS)
universe["sources"] = universe.apply(lambda r: "+".join(s for s in "ABC" if r[f"src_{s}"]), axis=1)

print(f"Candidate universe: {len(universe)} genes")
display(universe["sources"].value_counts().to_frame("n"))

In [ ]:
# --- Annotation 1 (Open Targets): tractability, chemical probes, drugs ---------------------
ANNOT_Q = '''
query annot($ids: [String!]!) {
  targets(ensemblIds: $ids) {
    id
    tractability { modality value }
    chemicalProbes { id isHighQuality }
    drugAndClinicalCandidates { count }
  }
}'''

ann, ids = [], universe["ensembl_id"].tolist()
for i in range(0, len(ids), 50):
    for t in ot_query(ANNOT_Q, {"ids": ids[i:i + 50]})["targets"]:
        pos = [x["modality"] for x in t["tractability"] if x["value"]]
        ann.append({"ensembl_id": t["id"],
                    "n_tract_positive": len(pos),
                    "tract_modalities": ";".join(sorted(set(pos))),
                    "n_chem_probes": len(t["chemicalProbes"]),
                    "n_chem_probes_hq": sum(p["isHighQuality"] for p in t["chemicalProbes"]),
                    "n_drugs_clinical": t["drugAndClinicalCandidates"]["count"]})
    print(f"  annotated {min(i + 50, len(ids))}/{len(ids)}")

universe = universe.merge(pd.DataFrame(ann), on="ensembl_id", how="left")
# The 2025 exam criterion, now explicit: no drugs, no chemical probe, <= 3 positive tractability assessments
universe["exam_tdark"] = ((universe["n_drugs_clinical"] == 0) & (universe["n_chem_probes"] == 0)
                          & (universe["n_tract_positive"] <= 3))
print("Exam-criterion T-dark:", int(universe["exam_tdark"].sum()), "of", len(universe))

In [ ]:
# --- Annotation 2 (Pharos/IDG): Target Development Level ----------------------------------
PHAROS_Q = "query t($q: ITarget!) { target(q: $q) { sym tdl fam novelty } }"
cache_path = RAW / "pharos_cache.json"
pharos = json.loads(cache_path.read_text()) if cache_path.exists() else {}

def pharos_target(sym, retries=3):
    for attempt in range(retries):
        try:
            r = requests.post(PHAROS_URL, json={"query": PHAROS_Q, "variables": {"q": {"sym": sym}}}, timeout=60)
            r.raise_for_status()
            return (r.json().get("data") or {}).get("target")
        except (requests.RequestException, ValueError) as e:
            if attempt == retries - 1:
                print(f"  {sym}: failed ({e})"); return None
            time.sleep(2)

todo = [s for s in universe["symbol"] if s not in pharos]
print(f"Querying Pharos for {len(todo)} genes ({len(pharos)} cached)...")
for k, sym in enumerate(todo, 1):
    pharos[sym] = pharos_target(sym)
    time.sleep(0.1)
    if k % 50 == 0:
        cache_path.write_text(json.dumps(pharos)); print(f"  {k}/{len(todo)}")
cache_path.write_text(json.dumps(pharos))

universe["tdl"] = universe["symbol"].map(lambda s: (pharos.get(s) or {}).get("tdl") or "not found")
universe["pharos_family"] = universe["symbol"].map(lambda s: (pharos.get(s) or {}).get("fam"))
universe["pharos_novelty"] = universe["symbol"].map(lambda s: (pharos.get(s) or {}).get("novelty"))
universe["understudied"] = universe["tdl"].isin(["Tdark", "Tbio"])
display(universe["tdl"].value_counts().to_frame("n"))

In [ ]:
# --- Save, summarise, plot ---------------------------------------------------------------
universe = universe.sort_values(["is_seed", "src_C", "src_B", "ot_overall_score"],
                                ascending=[False, False, False, False]).reset_index(drop=True)
universe.to_csv(PROC / "02_candidate_universe.csv", index=False)
(PROC / "02_manifest.json").write_text(json.dumps({
    "phase": "02_universe_and_darkness", "run_date": RUN_DATE, "disease_id": DISEASE_ID,
    "seeds": SEEDS, "causal_genetic_min": CAUSAL_GENETIC_MIN, "interaction_sources": INTERACTION_SOURCES,
    "max_siblings": MAX_SIBLINGS, "sister_diseases": siblings, "n_universe": len(universe),
    "pharos_queried_on": RUN_DATE}, indent=2))

TDL_ORDER = ["Tdark", "Tbio", "Tchem", "Tclin", "not found"]
by_src = pd.DataFrame({s: universe.loc[universe[f"src_{s}"], "tdl"].value_counts().reindex(TDL_ORDER, fill_value=0)
                       for s in "ABC"}).T
print("Development level by source (a gene can belong to more than one source):")
display(by_src)
print("Exam criterion vs Pharos level:")
display(pd.crosstab(universe["exam_tdark"], universe["tdl"]).reindex(columns=TDL_ORDER, fill_value=0))

cols = ["symbol", "sources", "tdl", "exam_tdark", "n_drugs_clinical", "n_chem_probes", "n_tract_positive",
        "evidence_types", "b_sources", "c_diseases"]
print("Sources B and C (the new candidates):")
display(universe.loc[universe["src_B"] | universe["src_C"], cols])
print("The 2025 exam genes:")
display(universe.loc[universe["symbol"].isin(EXAM_TDARK), cols])

# Figure: share of each development level per source (Tdark = darkest)
COLORS = {"Tdark": "#0F3D5C", "Tbio": "#3A78A6", "Tchem": "#86AECD", "Tclin": "#D3E0EB", "not found": "#FFFFFF"}
labels = {"A": "A · disease-associated", "B": "B · mechanistic neighbours", "C": "C · sister-disease genes"}
share = by_src.div(by_src.sum(axis=1).replace(0, 1), axis=0)
fig, ax = plt.subplots(figsize=(9, 3.2))
for row, s in enumerate(["C", "B", "A"]):
    left = 0
    for lvl in TDL_ORDER:
        w = share.loc[s, lvl]
        if w <= 0: continue
        nf = lvl == "not found"
        ax.barh(row, w, left=left, color=COLORS[lvl], height=0.6, hatch="//" if nf else None,
                edgecolor="#6B7280" if nf else "white", linewidth=0.8 if nf else 2)
        if w >= 0.08:
            ax.text(left + w / 2, row, f"{by_src.loc[s, lvl]}", ha="center", va="center", fontsize=9,
                    color="white" if lvl in ("Tdark", "Tbio") else "#1F2937",
                    bbox=dict(facecolor="white", edgecolor="none", pad=1) if nf else None)
        left += w
ax.set_yticks(range(3), [f"{labels[s]}  (n={int(by_src.loc[s].sum())})" for s in ["C", "B", "A"]])
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=COLORS[l], edgecolor="#6B7280" if l == "not found" else COLORS[l],
                         hatch="//" if l == "not found" else None, label=l) for l in TDL_ORDER],
          ncol=5, loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False, fontsize=9)
ax.set_title("Target development level (Pharos/IDG) by candidate source", fontsize=10, pad=28)
fig.tight_layout(); fig.savefig(FIG / "02_tdl_by_source.png", dpi=200); plt.show()

## Phase 2b — Evidence tiers, positional check, mechanism class (automatic)

Candidates are tiered by evidence quality: seed → sister disease → curated interaction → STRING-only → disease association with genetic or clinical support → background.

**Positional check.** Genetic evidence on a gene within 1 Mb of a seed may come from the seed locus itself rather than from the gene. Such genes are flagged automatically.

**Opportunity map.** The x axis is Pharos novelty (log; right means less studied), the y axis the level of chemical matter. The lower-right quadrant — understudied and unliganded — is where genuinely dark opportunities sit. Separating these two axes is the point: "no known drug" and "poorly understood" are different properties.

In [ ]:
# --- Phase 2b configuration: automatic steps only ------------------------------------------------------
# Nothing here is a judgement call. Every manual decision lives in the declared-curation cell further down.
if "universe" not in globals():
    universe = pd.read_csv(PROC / "02_candidate_universe.csv")
    pharos = json.loads((RAW / "pharos_cache.json").read_text())
    SEEDS = json.loads((PROC / "02_manifest.json").read_text())["seeds"]
if "assoc" not in globals():
    assoc = pd.read_csv(PROC / "01_associated_targets.csv")

CURATED_DBS = {"intact", "signor", "reactome"}   # B partners supported by any of these = "curated" tier
POSITIONAL_WINDOW_BP = 1_000_000                 # genetic evidence this close to a seed may be positional
print(f"Universe: {len(universe)} genes")

In [ ]:
# --- Extra annotation (Open Targets): UniProt, ChEMBL class, location, genomic position -------
EXTRA_Q = '''
query extra($ids: [String!]!) {
  targets(ensemblIds: $ids) {
    id
    proteinIds { id source }
    targetClass { label level }
    subcellularLocations { location }
    genomicLocation { chromosome start end }
  }
}'''

extra, ids = [], universe["ensembl_id"].tolist()
for i in range(0, len(ids), 50):
    for t in ot_query(EXTRA_Q, {"ids": ids[i:i + 50]})["targets"]:
        prot = t["proteinIds"]
        swiss = [p["id"] for p in prot if "swiss" in p["source"].lower()]
        anyup = [p["id"] for p in prot if "uniprot" in p["source"].lower()]
        gl = t["genomicLocation"] or {}
        extra.append({"ensembl_id": t["id"],
                      "uniprot": (swiss or anyup or [None])[0],
                      "chembl_class": "; ".join(sorted({c["label"] for c in t["targetClass"] if c["level"] in ("l1", "l2")})),
                      "locations": "; ".join(sorted({l["location"] for l in t["subcellularLocations"]})),
                      "chrom": gl.get("chromosome"), "start": gl.get("start"), "end": gl.get("end")})
universe = universe.drop(columns=[c for c in ["uniprot", "chembl_class", "locations", "chrom", "start", "end"]
                                  if c in universe.columns]).merge(pd.DataFrame(extra), on="ensembl_id", how="left")
print("UniProt found for", universe["uniprot"].notna().sum(), "of", len(universe))

In [ ]:
# --- Pharos retry by UniProt for symbols Pharos did not recognise ------------------------------
PHAROS_UP_Q = "query t($q: ITarget!) { target(q: $q) { sym tdl fam novelty } }"
missing = universe.loc[universe["tdl"].eq("not found") & universe["uniprot"].notna(), ["symbol", "uniprot"]]
print(f"Retrying {len(missing)} genes by UniProt:", ", ".join(missing["symbol"]))
for sym, acc in missing.itertuples(index=False):
    try:
        r = requests.post(PHAROS_URL, json={"query": PHAROS_UP_Q, "variables": {"q": {"uniprot": acc}}}, timeout=60)
        r.raise_for_status()
        hit = (r.json().get("data") or {}).get("target")
        if hit:
            pharos[sym] = hit
            print(f"  {sym} ({acc}) -> {hit['sym']} · {hit['tdl']}")
    except (requests.RequestException, ValueError) as e:
        print(f"  {sym}: failed ({e})")
    time.sleep(0.1)
(RAW / "pharos_cache.json").write_text(json.dumps(pharos))

universe["tdl"] = universe["symbol"].map(lambda s: (pharos.get(s) or {}).get("tdl") or "not found")
universe["pharos_family"] = universe["symbol"].map(lambda s: (pharos.get(s) or {}).get("fam"))
universe["pharos_novelty"] = universe["symbol"].map(lambda s: (pharos.get(s) or {}).get("novelty"))
print("Still not found:", ", ".join(universe.loc[universe["tdl"].eq("not found"), "symbol"]) or "none")

In [ ]:
# --- Evidence tiers, positional check, shortlist pool -------------------------------------------
dt = assoc[["ensembl_id"] + [c for c in assoc.columns if c.startswith("dt_")]]
universe = universe.drop(columns=[c for c in universe.columns if c.startswith("dt_")]).merge(dt, on="ensembl_id", how="left")
def col(name): return universe.get(name, pd.Series(0.0, index=universe.index)).fillna(0.0)

gscore = np.maximum(col("dt_genetic_association"), col("dt_genetic_literature"))
genetic = gscore > 0
clinical = col("dt_clinical") > 0

seed_pos = universe.loc[universe["is_seed"], ["chrom", "start", "end"]].dropna()
def seed_distance(r):
    d = [max(0, max(r["start"], s["start"]) - min(r["end"], s["end"]))
         for _, s in seed_pos.iterrows() if r["chrom"] == s["chrom"]]
    return min(d) if d else np.nan
universe["dist_to_seed_bp"] = universe.apply(lambda r: seed_distance(r) if pd.notna(r["start"]) else np.nan, axis=1)
universe["positional_artefact"] = (genetic & ~universe["is_seed"]
                                   & (universe["dist_to_seed_bp"] <= POSITIONAL_WINDOW_BP))

b_curated = universe["b_sources"].fillna("").apply(lambda s: bool(set(s.split(";")) & CURATED_DBS))
a_strong = (genetic & ~universe["positional_artefact"]) | clinical

def tier(i):
    if universe.at[i, "is_seed"]: return "seed"
    if universe.at[i, "src_C"]:   return "C · sister disease"
    if b_curated[i]:              return "B · curated"
    if universe.at[i, "src_B"]:   return "B · STRING only"
    if a_strong[i]:               return "A · genetic/clinical"
    return "A · background"
universe["tier"] = [tier(i) for i in universe.index]
universe["a_clinical_only"] = clinical & ~genetic & ~universe["src_B"] & ~universe["src_C"]


display(universe["tier"].value_counts().to_frame("n"))
universe["positional_note"] = universe["dist_to_seed_bp"].map(
    lambda d: f"genetic evidence within {d/1e3:.0f} kb of a seed gene").where(universe["positional_artefact"])
print("\nFlagged by the positional check:")
display(universe.loc[universe["positional_artefact"], ["symbol", "tier", "positional_note"]])

In [ ]:
# --- Mechanism class (automatic first pass + declared manual overrides) ---------------------------
def auto_class(r):
    if r["is_seed"]: return "causal gene"
    cls, fam, loc = str(r["chembl_class"]).lower(), str(r["pharos_family"]).lower(), str(r["locations"]).lower()
    if "ion channel" in cls or fam == "ic": return "ion channel"
    if any(k in cls for k in ("transcription factor", "epigenetic")) or fam.startswith(("tf", "nr", "epigenetic")):
        return "transcriptional / developmental regulator"
    if "membrane receptor" in cls or fam == "gpcr" or (fam == "kinase" and "cell membrane" in loc):
        return "membrane receptor"
    if "secreted" in cls or "secreted" in loc: return "secreted ligand"
    if any(k in cls for k in ("enzyme", "kinase", "cytosolic", "adaptor")) or fam in ("kinase", "enzyme"):
        return "intracellular signalling"
    return "other / unclassified"

universe["mech_class_auto"] = universe.apply(auto_class, axis=1)

CHEM_LEVELS = ["none", "predicted tractable", "chemical probe", "clinical candidate"]
universe["chem_level"] = np.select(
    [universe["n_drugs_clinical"] > 0, universe["n_chem_probes"] > 0, universe["n_tract_positive"] > 0],
    [3, 2, 1], default=0)
universe["chem_label"] = universe["chem_level"].map(dict(enumerate(CHEM_LEVELS)))
print("Automatic mechanism classes (reviewed in the declared-curation step):")
display(universe["mech_class_auto"].value_counts().to_frame("n_genes"))

In [ ]:
# --- Save the automatic stage -------------------------------------------------------------------------
universe.to_csv(PROC / "02b_universe_tiered.csv", index=False)
(PROC / "02b_manifest.json").write_text(json.dumps({
    "phase": "02b_tiers_and_classes_automatic", "run_date": RUN_DATE,
    "curated_dbs": sorted(CURATED_DBS), "positional_window_bp": POSITIONAL_WINDOW_BP,
    "n_universe": len(universe), "n_positional_flagged": int(universe["positional_artefact"].sum()),
    "tier_counts": universe["tier"].value_counts().to_dict()}, indent=2))
print("Saved 02b_universe_tiered.csv -- the declared-curation step reads it from disk,")
print("so the effect of curation can be diffed against this file.")

## Phase 2c — Declared curation

Automatic steps cannot judge every biological relationship. All manual decisions are made **here, in one place, each with a written reason**, reading the Phase 2b output and writing new files. Nothing upstream is modified, so the effect of curation stays visible and reversible — and a reader can disagree with a specific line rather than with the pipeline.

Declared here: partners of oncogenic *NTRK1* fusions and ligands of unrelated receptor tyrosine kinases linked only by STRING (flagged out of the pool); a minimum genetic score for genes supported by source A alone; and mechanism classes grounded in TrkA biology rather than in generic protein-family labels.

In [ ]:
# --- Declared curation layer, applied on top of the automatic Phase 2b output ------------------------
# Nothing above is modified: this step reads 02b, applies declared choices, writes new 02c files.
universe = pd.read_csv(PROC / "02b_universe_tiered.csv")
SEEDS = json.loads((PROC / "02_manifest.json").read_text())["seeds"]
_m1 = json.loads((PROC / "01_manifest.json").read_text())
DISEASE_ID, DISEASE_NAME = _m1["disease_id"], _m1["disease_name"]

A_GENETIC_MIN = 0.1   # minimum genetic score for an A-only gene to enter the pool

OTHER_RTK = "ligand of a different, non-neurotrophic receptor tyrosine kinase ({}); STRING-only association"
FUSION = "partner in an oncogenic {}-NTRK1 fusion (cancer literature), not a physiological interactor"
CURATION_FLAGS = {   # full list: the 8 flags of Phase 2b + 9 added at review (8 Oct 2026)
    "TPM3": FUSION.format("TPM3"), "TPR": FUSION.format("TPR"), "TFG": FUSION.format("TFG"),
    "KITLG": OTHER_RTK.format("KIT"), "HGF": OTHER_RTK.format("MET"), "EGF": OTHER_RTK.format("EGFR"),
    "ANGPT1": OTHER_RTK.format("TEK"), "ANGPT2": OTHER_RTK.format("TEK"), "INS": OTHER_RTK.format("INSR"),
    "IGF1": OTHER_RTK.format("IGF1R"), "IGF2": OTHER_RTK.format("IGF1R/INSR"), "CSF1": OTHER_RTK.format("CSF1R"),
    "FGF2": OTHER_RTK.format("FGFRs"), "FGF8": OTHER_RTK.format("FGFRs"), "FLT3LG": OTHER_RTK.format("FLT3"),
    "PGF": OTHER_RTK.format("FLT1"), "EFNA5": OTHER_RTK.format("Eph receptors"),
}
_CLASSES = {   # declared mechanism classes (reviewed 8 Oct 2026); unlisted genes keep the automatic class
    "neurotrophin ligand / co-receptor": "NGF NTF3 NTF4 BDNF GDNF NGFR SORT1",
    "parallel receptor": "NTRK2",
    "GPCR transactivation route": "ADORA2A ADCYAP1 ADCYAP1R1",
    "docking adaptor": "SHC1 SHC2 SHC3 SHC4 FRS2 GRB2 IRS1 IRS2 SH2B1 SH2B2 SH2B3 CRK CRKL APPL1 KIDINS220 RAPGEF1 YWHAB",
    "signalling effector": ("PIK3CA PIK3CB PIK3CD PIK3R1 PIK3R2 PIK3R3 PLCG1 PLCG2 SRC FYN ABL1 BRAF MATK "
                            "RAP1A RIT1 RIT2 PTPN1 PTPN6 DUSP26 CBLB ARHGAP32"),
    "trafficking / signalling endosome": "AP2A1 AP2A2 AP2B1 AP2M1 AP2S1 CLTA CLTC SH3GL2 RAB7A DNAL4",
    "transcriptional / developmental regulator": "PRDM12 ZFHX2 STAT3 CTNNB1",
    "ion channel": "SCN9A SCN11A",
    "symptomatic (autonomic)": "ADRA1A ADRA1B ADRA1D ADRA2A ADRA2B ADRA2C ADRB1 ADRB2 ADRB3 DDC",
    "other": "HSP90AB1 CCN2",
}
MANUAL_CLASS = {g: c for c, genes in _CLASSES.items() for g in genes.split()}

def col(n): return universe[n].fillna(0.0) if n in universe.columns else pd.Series(0.0, index=universe.index)
flag = lambda n: universe[n].fillna(False).astype(bool)
gscore = np.maximum(col("dt_genetic_association"), col("dt_genetic_literature"))
genetic, genetic_ok, clinical = gscore > 0, gscore >= A_GENETIC_MIN, col("dt_clinical") > 0
pos_art = flag("positional_artefact")

was_in_pool = (flag("in_pool") if "in_pool" in universe.columns
               else ~universe["tier"].eq("A · background"))   # automatic tiers alone
is_A = universe["tier"].str.startswith("A ·")
a_strong = (genetic_ok & ~pos_art) | clinical
universe.loc[is_A, "tier"] = np.where(a_strong[is_A], "A · genetic/clinical", "A · background")

weak = genetic & ~genetic_ok & ~pos_art & ~clinical & ~flag("src_B") & ~flag("src_C")
notes = universe["symbol"].map(CURATION_FLAGS)
notes = notes.where(notes.notna(), universe["dist_to_seed_bp"].map(
    lambda d: f"genetic evidence within {d/1e3:.0f} kb of a seed gene: possible positional artefact").where(pos_art))
notes = notes.where(notes.notna(), gscore.map(
    lambda g: f"genetic evidence below the declared minimum ({g:.3f} < {A_GENETIC_MIN})").where(weak))
universe["curation_note"] = notes
universe["in_pool"] = ~universe["tier"].eq("A · background") & ~universe["symbol"].isin(CURATION_FLAGS)
universe["mech_class"] = [MANUAL_CLASS.get(s, a) for s, a in zip(universe["symbol"], universe["mech_class_auto"])]
universe["class_source"] = np.where(universe["symbol"].isin(MANUAL_CLASS), "manual", "auto")

TIER_ORDER = ["seed", "C · sister disease", "B · curated", "B · STRING only", "A · genetic/clinical"]
pool = (universe.loc[universe["in_pool"]]
          .assign(_t=lambda d: d["tier"].map({t: i for i, t in enumerate(TIER_ORDER)}))
          .sort_values(["_t", "mech_class", "symbol"]).drop(columns="_t"))

universe.to_csv(PROC / "02c_universe_curated.csv", index=False)
pool[["symbol", "tier", "mech_class", "class_source", "tdl", "pharos_novelty", "chem_label", "n_drugs_clinical"]] \
    .to_csv(PROC / "02c_pool_curated.csv", index=False)
(PROC / "02c_manifest.json").write_text(json.dumps({
    "phase": "02c_declared_curation", "run_date": RUN_DATE, "input": "02b_universe_tiered.csv",
    "a_genetic_min": A_GENETIC_MIN, "curation_flags": CURATION_FLAGS, "manual_class": MANUAL_CLASS,
    "n_pool_before": int(was_in_pool.sum()), "n_pool_after": int(universe["in_pool"].sum())}, indent=2))

print(f"Pool: {int(was_in_pool.sum())} from the automatic tiers -> {int(universe['in_pool'].sum())} after declared curation")
print("\nRemoved by the declared curation, each with its reason:")
display(universe.loc[was_in_pool & ~universe["in_pool"], ["symbol", "tier", "curation_note"]])
print("Pool by mechanism class and tier:")
display(pd.crosstab(pool["mech_class"], pool["tier"]).reindex(columns=TIER_ORDER, fill_value=0))
unclassified = pool.loc[pool["class_source"].eq("auto") & ~pool["is_seed"], ["symbol", "tier", "mech_class"]]
print("Pool genes still on the automatic class:", ", ".join(unclassified["symbol"]) or "none")

In [ ]:
# --- Opportunity map, curated pool ------------------------------------------------------------------
TIER_STYLE = {   # 3 validated hues; STRING-only shares the B hue with a hollow marker
    "C · sister disease":   dict(color="#eb6834", filled=True),
    "B · curated":          dict(color="#2a78d6", filled=True),
    "B · STRING only":      dict(color="#2a78d6", filled=False),
    "A · genetic/clinical": dict(color="#1baf7a", filled=True)}
CHEM_LEVELS = ["none", "predicted tractable", "chemical probe", "clinical candidate"]
LABEL_ALWAYS = set(EXAM_TDARK) | set(universe.loc[flag("src_C"), "symbol"])

m = pool.dropna(subset=["pharos_novelty"]).copy()
rng = np.random.default_rng(SEED)
m["y"] = m["chem_level"] + rng.uniform(-0.28, 0.28, len(m))
x_split = m.loc[~m["is_seed"].astype(bool), "pharos_novelty"].median()

fig, ax = plt.subplots(figsize=(10, 6.2))
ax.axvline(x_split, color="#c3c2b7", lw=1, ls="--", zorder=0)
ax.axhline(1.5, color="#c3c2b7", lw=1, ls="--", zorder=0)
for t, st in TIER_STYLE.items():
    d = m[m["tier"] == t]
    ax.scatter(d["pharos_novelty"], d["y"], s=46, lw=1.4, zorder=3,
               facecolors=st["color"] if st["filled"] else "white", edgecolors=st["color"], label=f"{t} (n={len(d)})")
s = m[m["is_seed"].astype(bool)]
ax.scatter(s["pharos_novelty"], s["y"], marker="*", s=260, color="#0b0b0b", zorder=4, label="causal gene")
for _, r in m.iterrows():
    if r["is_seed"] or r["symbol"] in LABEL_ALWAYS or (r["tier"] == "B · curated" and r["b_n_sources"] >= 2):
        ax.annotate(r["symbol"], (r["pharos_novelty"], r["y"]), xytext=(5, 4), textcoords="offset points",
                    fontsize=8, color="#0b0b0b", fontweight="bold" if r["symbol"] in EXAM_TDARK else "normal")
ax.set_xscale("log"); ax.set_yticks(range(4), CHEM_LEVELS); ax.set_ylim(-0.6, 3.6)
ax.set_xlabel("Pharos novelty (log)   ←  well studied  ·  understudied  →", color="#52514e")
ax.set_ylabel("Chemical matter", color="#52514e")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.text(0.99, 0.02, "understudied, no chemical matter\n(T-dark opportunity zone)", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=8, color="#52514e", style="italic")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8)
ax.set_title(f"Opportunity map · {DISEASE_NAME} · curated pool (bold = 2025 exam genes)", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "02c_opportunity_map_curated.png", dpi=200); plt.show()

## Phase 3 — Causal-gene variants: which strategy fits which genotype

A strategy only makes sense for patients whose variant leaves room for it. ClinVar variants for the causal gene are restricted to one reference transcript, classified as null / missense / in-frame, and mapped onto protein regions from UniProt. AlphaMissense scores every possible substitution.

**Consistency check:** the reference residue of every missense variant is compared with the UniProt sequence before any of it is trusted.

**Caveat:** counts are *variants*, not *patients*. Allele frequencies, including founder variants, are not modelled.

In [ ]:
# --- Phase 3 configuration + protein reference (UniProt): sequence, topology, domains ----------
import re
if "universe" not in globals() or "class_source" not in universe.columns:
    universe = pd.read_csv(PROC / "02c_universe_curated.csv")
    SEEDS = json.loads((PROC / "02_manifest.json").read_text())["seeds"]

SEED_SYMBOL = list(SEEDS.values())[0]
SEED_UNIPROT = universe.loc[universe["symbol"] == SEED_SYMBOL, "uniprot"].iloc[0]
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
AM_LIKELY_PATHOGENIC = 0.564   # AlphaMissense published class thresholds
AM_LIKELY_BENIGN = 0.34
print(f"Seed: {SEED_SYMBOL} · UniProt {SEED_UNIPROT}")

up = requests.get(f"https://rest.uniprot.org/uniprotkb/{SEED_UNIPROT}.json", timeout=60)
up.raise_for_status(); up = up.json()
SEQ = up["sequence"]["value"]
(RAW / f"uniprot_{SEED_UNIPROT}.json").write_text(json.dumps(up))

KEEP_FT = {"Signal", "Topological domain", "Transmembrane", "Domain", "Repeat"}
feats = []
for f in up.get("features", []):
    s, e = f["location"]["start"].get("value"), f["location"]["end"].get("value")
    if f["type"] in KEEP_FT and s and e:
        feats.append({"type": f["type"], "description": f.get("description", ""), "start": s, "end": e})
feats = pd.DataFrame(feats)
display(feats)

def region_of(pos):
    # Coarse, strategy-relevant region for a residue position
    if pd.isna(pos): return "no protein position"
    pos = int(pos)
    here = feats[(feats["start"] <= pos) & (feats["end"] >= pos)]
    if here["description"].str.contains("kinase", case=False).any(): return "kinase domain"
    if (here["type"] == "Signal").any(): return "signal peptide"
    if (here["type"] == "Transmembrane").any(): return "transmembrane"
    topo = " ".join(here.loc[here["type"] == "Topological domain", "description"]).lower()
    if "extracellular" in topo: return "extracellular (ligand binding)"
    if "cytoplasmic" in topo: return "intracellular, outside kinase"
    return "unassigned"
print(f"Sequence length: {len(SEQ)} aa")

In [ ]:
# --- ClinVar via NCBI E-utilities: all variants in the seed gene -----------------------------------
es = requests.get(f"{EUTILS}/esearch.fcgi", params={"db": "clinvar", "term": f"{SEED_SYMBOL}[gene]",
                                                    "retmax": 10000, "retmode": "json"}, timeout=60)
es.raise_for_status()
cv_ids = es.json()["esearchresult"]["idlist"]
print(f"ClinVar records for {SEED_SYMBOL}[gene]: {len(cv_ids)}")

docs = []
for i in range(0, len(cv_ids), 200):
    r = requests.post(f"{EUTILS}/esummary.fcgi", data={"db": "clinvar", "id": ",".join(cv_ids[i:i + 200]),
                                                       "retmode": "json"}, timeout=120)
    r.raise_for_status()
    res = r.json()["result"]
    docs += [res[u] for u in res.get("uids", [])]
    time.sleep(0.4)                                   # NCBI limit without API key: 3 requests/s
(RAW / f"clinvar_{SEED_SYMBOL}_{RUN_DATE}.json").write_text(json.dumps(docs))

def classification(d):
    for k in ("germline_classification", "clinical_significance"):
        v = d.get(k)
        if isinstance(v, dict) and v.get("description"):
            return v["description"]
    return "not provided"

cv = pd.DataFrame([{
    "clinvar_id": d.get("uid"), "accession": d.get("accession"), "title": d.get("title", ""),
    "obj_type": d.get("obj_type", ""), "classification": classification(d),
    "consequences": ";".join(d.get("molecular_consequence_list") or []),
    "genes": ";".join(g.get("symbol", "") for g in d.get("genes") or [])} for d in docs])
cv = cv[cv["genes"] == SEED_SYMBOL].copy()            # single-gene variants only (no multi-gene CNVs)
print(f"Single-gene {SEED_SYMBOL} variants: {len(cv)}")
display(cv["classification"].value_counts().head(12).to_frame("n"))
if len(cv): print("Example record keys:", sorted(docs[0].keys()))

In [ ]:
# --- Variant classes (null / missense / in-frame / other) and protein region -----------------------
AA3 = {"Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H",
       "Ile": "I", "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S", "Thr": "T", "Trp": "W",
       "Tyr": "Y", "Val": "V"}

def parse(row):
    t = row["title"]
    tx = re.match(r"^(N[MR]_\d+\.\d+)\(", t)
    p = re.search(r"\(p\.([^)\s]+)\)", t)
    pstr = p.group(1) if p else ""
    pos = re.search(r"(\d+)", pstr)
    pos = int(pos.group(1)) if pos else np.nan
    cons = row["consequences"].lower()
    canonical_splice = re.search(r"c\.\d+[+-][12][ACGT]?>", t) or "splice donor" in cons or "splice acceptor" in cons
    if "fs" in pstr:                                         vclass = "null · frameshift"
    elif pstr.endswith(("Ter", "*")):                         vclass = "null · nonsense"
    elif re.fullmatch(r"Met1(\?|[A-Z][a-z]{2})", pstr):      vclass = "null · start loss"
    elif not pstr and canonical_splice:                      vclass = "null · canonical splice"
    elif any(k in pstr for k in ("del", "dup", "ins")):      vclass = "in-frame indel"
    elif pstr.endswith("="):                                 vclass = "synonymous"
    elif re.fullmatch(r"[A-Z][a-z]{2}\d+[A-Z][a-z]{2}", pstr): vclass = "missense"
    else:                                                    vclass = "other / non-coding"
    ref = alt = None
    if vclass == "missense":
        ref, alt = AA3.get(pstr[:3]), AA3.get(pstr[-3:])
        if ref == alt: vclass = "synonymous"
    return pd.Series({"transcript": tx.group(1) if tx else None, "p_hgvs": pstr, "aa_pos": pos,
                      "ref_aa": ref, "alt_aa": alt, "vclass": vclass})

cv = pd.concat([cv, cv.apply(parse, axis=1)], axis=1)
cls = cv["classification"].str.lower()
cv["group"] = np.select(
    [cls.str.contains("conflicting"), cls.str.contains("pathogenic") & ~cls.str.contains("benign"),
     cls.str.contains("uncertain"), cls.str.contains("benign")],
    ["conflicting", "P/LP", "VUS", "B/LB"], default="other")

# Keep the reference transcript (most frequent among P/LP) so positions match the UniProt sequence
REF_TX = cv.loc[cv["group"] == "P/LP", "transcript"].mode().iloc[0]
cv_tx = cv[cv["transcript"] == REF_TX].copy()
mis = cv_tx[cv_tx["vclass"] == "missense"]
match = (mis.apply(lambda r: 0 < r["aa_pos"] <= len(SEQ) and SEQ[int(r["aa_pos"]) - 1] == r["ref_aa"], axis=1)).mean()
print(f"Reference transcript: {REF_TX} · {len(cv_tx)} variants · "
      f"missense reference residue matches UniProt in {match:.0%} of cases")
if match < 0.95:
    print("WARNING: transcript and UniProt isoform may differ -- positions need re-mapping before use.")

cv_tx["region"] = cv_tx["aa_pos"].map(region_of)

plp = cv_tx[cv_tx["group"] == "P/LP"]
print(f"\nPathogenic / likely pathogenic on {REF_TX}: {len(plp)}")
display(plp["vclass"].value_counts().to_frame("n"))
display(pd.crosstab(plp["vclass"], plp["region"]))

In [ ]:
# --- AlphaMissense (via AlphaFold DB): scores for every possible substitution -------------------------
afdb = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{SEED_UNIPROT}", timeout=60)
afdb.raise_for_status(); afdb = afdb.json()[0]
print("AlphaFold DB entry:", afdb.get("entryId"), "· model sequence length:", len(afdb.get("uniprotSequence", "")))
am_url = afdb.get("amAnnotationsUrl")
if not am_url:
    raise RuntimeError("No AlphaMissense URL in the AlphaFold DB entry; keys: " + ", ".join(sorted(afdb)))
am = pd.read_csv(am_url)
am.to_csv(RAW / f"alphamissense_{SEED_UNIPROT}.csv", index=False)
am["aa_pos"] = am["protein_variant"].str[1:-1].astype(int)
am["ref_aa"], am["alt_aa"] = am["protein_variant"].str[0], am["protein_variant"].str[-1]
if afdb.get("uniprotSequence") and afdb["uniprotSequence"] != SEQ:
    print("WARNING: AlphaFold DB sequence differs from the current UniProt sequence.")

# Score each ClinVar missense on the reference transcript
cv_tx = cv_tx.drop(columns=[c for c in ("am_pathogenicity", "am_class") if c in cv_tx.columns]).merge(
    am[["aa_pos", "ref_aa", "alt_aa", "am_pathogenicity", "am_class"]], on=["aa_pos", "ref_aa", "alt_aa"], how="left")
scored = cv_tx[(cv_tx["vclass"] == "missense") & cv_tx["am_class"].notna()]
print("\nAlphaMissense class of ClinVar missense variants, by ClinVar group:")
display(pd.crosstab(scored["group"], scored["am_class"]).reindex(["P/LP", "VUS", "B/LB", "conflicting"]).dropna(how="all"))

# Per-residue profile: mean pathogenicity of all 19 substitutions at each position
am_res = am.groupby("aa_pos")["am_pathogenicity"].mean().rename("am_mean")
am_res.to_csv(PROC / "03_alphamissense_per_residue.csv")
reg_profile = (am_res.reset_index().assign(region=lambda d: d["aa_pos"].map(region_of))
                 .groupby("region")["am_mean"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False))
print("Mean AlphaMissense pathogenicity by region:")
display(reg_profile)

In [ ]:
# --- Which therapeutic strategies apply to which variant classes ------------------------------------
def allele_group(r):
    if r["vclass"].startswith("null"): return "null (no functional protein expected)"
    if r["vclass"] == "missense":
        return {"extracellular (ligand binding)": "missense · extracellular",
                "kinase domain": "missense · kinase domain"}.get(r["region"], "missense · other region")
    if r["vclass"] == "in-frame indel": return "in-frame indel"
    return "other / non-coding"

STRATEGY_FIT = {   # declared reasoning; 'possible' always requires experimental confirmation
    "null (no functional protein expected)": {"restore (replace/correct gene)": "yes", "via TrkA (ligand, PRDM12)": "no",
        "GPCR transactivation of TrkA": "no", "TrkA-independent (downstream / parallel)": "yes", "symptomatic": "yes"},
    "missense · extracellular": {"restore (replace/correct gene)": "yes", "via TrkA (ligand, PRDM12)": "unlikely",
        "GPCR transactivation of TrkA": "possible, if the protein folds and traffics",
        "TrkA-independent (downstream / parallel)": "yes", "symptomatic": "yes"},
    "missense · kinase domain": {"restore (replace/correct gene)": "yes", "via TrkA (ligand, PRDM12)": "no",
        "GPCR transactivation of TrkA": "no", "TrkA-independent (downstream / parallel)": "yes", "symptomatic": "yes"},
    "missense · other region": {"restore (replace/correct gene)": "yes", "via TrkA (ligand, PRDM12)": "uncertain",
        "GPCR transactivation of TrkA": "uncertain", "TrkA-independent (downstream / parallel)": "yes", "symptomatic": "yes"},
    "in-frame indel": {"restore (replace/correct gene)": "yes", "via TrkA (ligand, PRDM12)": "uncertain",
        "GPCR transactivation of TrkA": "uncertain", "TrkA-independent (downstream / parallel)": "yes", "symptomatic": "yes"},
}
plp = cv_tx[cv_tx["group"] == "P/LP"].copy()
plp["allele_group"] = plp.apply(allele_group, axis=1)
counts = plp["allele_group"].value_counts()
fit = pd.DataFrame(STRATEGY_FIT).T.reindex(counts.index.intersection(list(STRATEGY_FIT)))
fit.insert(0, "n_variants", counts.reindex(fit.index).astype(int))
fit.insert(1, "share", (fit["n_variants"] / len(plp)).map("{:.0%}".format))
print(f"P/LP variants on {REF_TX}: {len(plp)} (counts are variants, not patients: allele frequencies are not modelled)")
display(fit)
fit.to_csv(PROC / "03_strategy_fit_by_allele_group.csv")

In [ ]:
# --- Figure: variant map along the protein with AlphaMissense profile ----------------------------------
COL = {"missense": "#2a78d6", "null": "#eb6834", "in-frame indel": "#1baf7a"}
plot_v = plp[plp["aa_pos"].notna()].copy()
plot_v["kind"] = np.where(plot_v["vclass"].str.startswith("null"), "null", plot_v["vclass"])
plot_v = plot_v[plot_v["kind"].isin(COL)]
n_nopos = int(plp["aa_pos"].isna().sum())

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 6.4), sharex=True,
                                    gridspec_kw={"height_ratios": [2.2, 0.6, 1.6], "hspace": 0.08})
# 1) lollipops: stacked count of P/LP variants per residue
for kind, c in COL.items():
    d = plot_v[plot_v["kind"] == kind].groupby("aa_pos").size()
    ax1.vlines(d.index, 0, d.values, color=c, lw=1.2, alpha=0.8)
    ax1.scatter(d.index, d.values, s=26, color=c, zorder=3, label=f"{kind} (n={int(d.sum())})")
from matplotlib.ticker import MaxNLocator
ymax = max(1, int(plot_v.groupby(["kind", "aa_pos"]).size().max())) if len(plot_v) else 1
ax1.set_ylabel("P/LP variants", color="#52514e"); ax1.set_ylim(0, ymax * 1.15)
ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
ax1.legend(loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False, fontsize=8, ncol=3)
ax1.set_title(f"{SEED_SYMBOL} ({SEED_UNIPROT}) · ClinVar pathogenic variants, protein regions, AlphaMissense",
              fontsize=9, pad=22)
if n_nopos:
    ax1.text(1, 1.02, f"{n_nopos} splice/other variants without a protein position not shown",
             transform=ax1.transAxes, ha="right", va="bottom", fontsize=7, color="#52514e")
# 2) region track: topology in light grey, domains in mid grey, labelled
for _, f in feats.iterrows():
    if f["type"] in ("Topological domain", "Transmembrane", "Signal"):
        ax2.add_patch(plt.Rectangle((f["start"], 0.35), f["end"] - f["start"] + 1, 0.3, color="#e1e0d9"))
for _, f in feats[feats["type"].isin(["Domain", "Transmembrane"])].iterrows():
    ax2.add_patch(plt.Rectangle((f["start"], 0.15), f["end"] - f["start"] + 1, 0.7,
                                color="#52514e" if "kinase" in f["description"].lower() else "#898781"))
    if f["end"] - f["start"] > 40:
        ax2.text((f["start"] + f["end"]) / 2, 0.5, f["description"] or f["type"], ha="center", va="center",
                 fontsize=7, color="white")
ax2.set_ylim(0, 1); ax2.set_yticks([])
for sp in ("left", "right", "top", "bottom"): ax2.spines[sp].set_visible(False)
# 3) AlphaMissense mean pathogenicity per residue
ax3.plot(am_res.index, am_res.values, color="#0b0b0b", lw=1)
ax3.axhline(AM_LIKELY_PATHOGENIC, color="#eb6834", lw=1, ls="--")
ax3.text(0.995, 0.93, "likely pathogenic threshold", transform=ax3.transAxes, ha="right", va="top",
         fontsize=7, color="#52514e", bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax3.set_ylim(0, 1); ax3.set_xlim(0, len(SEQ) + 1)
ax3.set_ylabel("AlphaMissense\n(mean per residue)", color="#52514e"); ax3.set_xlabel("residue", color="#52514e")
for a in (ax1, ax3):
    for sp in ("top", "right"): a.spines[sp].set_visible(False)
fig.savefig(FIG / "03_variant_map.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# --- Save Phase 3 outputs -------------------------------------------------------------------------
cv_tx.to_csv(PROC / "03_clinvar_variants_classified.csv", index=False)
(PROC / "03_manifest.json").write_text(json.dumps({
    "phase": "03_variants", "run_date": RUN_DATE, "seed": SEED_SYMBOL, "uniprot": SEED_UNIPROT,
    "reference_transcript": REF_TX, "n_clinvar_records": len(cv_ids), "n_plp_on_reference": int(len(plp)),
    "missense_reference_match": float(match), "am_thresholds": [AM_LIKELY_BENIGN, AM_LIKELY_PATHOGENIC],
    "alphafold_entry": afdb.get("entryId")}, indent=2))

## Phase 3b — What kind of edit would revert each variant

Read from the HGVS `c.` string only. A correction that is a transition is within the scope of base editing in principle; a transversion is not. Indels up to a declared cut-off are prime-editing territory; anything larger or unparseable is reported rather than silently dropped.

**Explicitly out of scope:** PAM sites, editing windows, guide design, off-target assessment, efficiency — the parts where a strategy is actually shown to work. This step stops at descriptive statistics over variant notation.

**The caveat that matters most:** correcting the sequence does not regenerate neurons already lost during development. These counts describe theoretical reach, not clinical usefulness after birth.

In [ ]:
# --- Phase 3b: what kind of edit would revert each pathogenic variant ---------------------------------
# Notation-level only: the required correction is read from the HGVS c. string.
# No PAM search, no guide design, no editing windows, no off-target or efficiency assessment.
import re

SUB_RE = re.compile(r"c\.[-*]?\d+(?:[+-]\d+)?([ACGT])>([ACGT])")                       # c.1729G>C, c.717+4A>T
RANGE_RE = re.compile(r"c\.[-*]?(\d+)(?:[+-]\d+)?_[-*]?(\d+)(?:[+-]\d+)?(delins|del|dup|ins)([ACGT]*)")
SINGLE_RE = re.compile(r"c\.[-*]?(\d+)(?:[+-]\d+)?(delins|del|dup|ins)([ACGT]*)")
REPEAT_RE = re.compile(r"c\.[-*]?\d+(?:_[-*]?\d+)?([ACGT]+)\[(\d+)\]")   # repeat allele, e.g. c.780GAA[1]
MAX_SMALL_INDEL_BP = 20        # declared cut-off between "small indel" and "large / complex"

# A base editor writes A->G (ABE) or C->T (CBE) on either strand, so correcting a variant needs:
BASE_EDITABLE = {("A", "G"): "ABE (A>G)", ("T", "C"): "ABE (A>G, antisense strand)",
                 ("C", "T"): "CBE (C>T)", ("G", "A"): "CBE (C>T, antisense strand)"}

def indel_size(start, end, kind, seq):
    # Bases the edit has to change; for ins/delins the inserted sequence is what counts.
    span = abs(end - start) + 1 if end is not None else 1
    if kind == "ins":
        return (len(seq) or None), f"insertion of {len(seq)} bp" if seq else "insertion of unstated length"
    if kind == "delins":
        n = max(span, len(seq))
        return n, f"{span} bp replaced by {len(seq) or 'unstated'} bp"
    return span, f"{kind} of {span} bp"

def correction_class(title):
    # Returns (category, detail) describing the edit that would restore the reference sequence.
    m = SUB_RE.search(title)
    if m:
        ref, alt = m.groups()
        fix = BASE_EDITABLE.get((alt, ref))                                 # correction is alt -> ref
        if fix:
            return "transition · base-editable in principle", f"{ref}>{alt}; reverting needs {fix}"
        return "transversion · outside base editing", f"{ref}>{alt}; reverting needs a transversion"
    m = REPEAT_RE.search(title)
    if m:
        unit = len(m.group(1))
        cat = "small indel · prime-editing territory" if unit <= MAX_SMALL_INDEL_BP else "large / complex"
        return cat, f"repeat allele, {unit} bp unit (change size not stated in the HGVS title)"
    m = RANGE_RE.search(title)
    if m:
        start, end, kind, seq = int(m.group(1)), int(m.group(2)), m.group(3), m.group(4)
    else:
        m = SINGLE_RE.search(title)
        if not m:
            return "not parsed from HGVS", ""
        start, end, kind, seq = int(m.group(1)), None, m.group(2), m.group(3)
    size, detail = indel_size(start, end, kind, seq)
    if size is None:
        return "not parsed from HGVS", detail
    if size <= MAX_SMALL_INDEL_BP:
        return "small indel · prime-editing territory", detail
    return "large / complex", f"{detail} (above the {MAX_SMALL_INDEL_BP} bp cut-off)"

plp = cv_tx[cv_tx["group"] == "P/LP"].copy()
plp[["correction_class", "correction_detail"]] = plp["title"].apply(
    lambda t: pd.Series(correction_class(t)))
plp["variant_kind"] = np.where(plp["vclass"].str.startswith("null"), "null",
                               np.where(plp["vclass"] == "missense", "missense", "in-frame / other"))

ORDER = ["transition · base-editable in principle", "transversion · outside base editing",
         "small indel · prime-editing territory", "large / complex", "not parsed from HGVS"]
summary = (plp["correction_class"].value_counts().reindex(ORDER).dropna().astype(int).to_frame("n_variants")
             .assign(share=lambda d: (d["n_variants"] / len(plp)).map("{:.0%}".format)))
print(f"P/LP variants on {REF_TX}: {len(plp)}  (variants, not patients: allele frequencies are not modelled)")
display(summary)
display(pd.crosstab(plp["correction_class"], plp["variant_kind"]).reindex(summary.index))
if (plp["correction_class"] == "not parsed from HGVS").any():
    print("Not parsed (check these by hand):")
    display(plp.loc[plp["correction_class"] == "not parsed from HGVS", ["accession", "title", "vclass"]].head(15))
plp.to_csv(PROC / "03b_correction_classes.csv", index=False)

In [ ]:
# --- Figure + declared caveats -------------------------------------------------------------------------
KIND_COL = {"null": "#2a78d6", "missense": "#eb6834", "in-frame / other": "#1baf7a"}
tab = pd.crosstab(plp["correction_class"], plp["variant_kind"]).reindex(summary.index).fillna(0)
tab = tab.reindex(columns=[k for k in KIND_COL if k in tab.columns])

fig, ax = plt.subplots(figsize=(9.5, 3.6))
left = np.zeros(len(tab))
for kind in tab.columns:
    ax.barh(range(len(tab)), tab[kind], left=left, color=KIND_COL[kind], height=0.62,
            edgecolor="white", linewidth=2, label=kind)
    for i, (v, l) in enumerate(zip(tab[kind], left)):
        if v >= max(1, 0.04 * len(plp)):
            ax.text(l + v / 2, i, int(v), ha="center", va="center", fontsize=9, color="white")
    left += tab[kind].to_numpy()
ax.set_yticks(range(len(tab)), [t.replace(" · ", "\n") for t in tab.index], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("pathogenic / likely pathogenic variants", color="#52514e")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
from matplotlib.ticker import MaxNLocator
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, fontsize=8)
ax.set_title(f"{SEED_SYMBOL} · edit required to revert each pathogenic variant (HGVS notation only)",
             fontsize=10, pad=24)
fig.tight_layout(); fig.savefig(FIG / "03b_correction_classes.png", dpi=200); plt.show()

CAVEATS = [
    "Notation-level analysis: no PAM site, editing window, guide RNA, off-target or efficiency assessment.",
    "'Base-editable in principle' means the required substitution is a transition, nothing more.",
    "Counts are variants, not patients: allele frequencies and founder variants are not modelled.",
    "Correcting the sequence does not regenerate neurons already lost during development, so this "
    "describes the theoretical reach of a gene-correction approach, not its clinical usefulness after birth.",
]
print("\nDeclared caveats (also in the README):")
for c in CAVEATS: print(" -", c)

(PROC / "03b_manifest.json").write_text(json.dumps({
    "phase": "03b_correction_classes", "run_date": RUN_DATE, "seed": SEED_SYMBOL,
    "reference_transcript": REF_TX, "max_small_indel_bp": MAX_SMALL_INDEL_BP,
    "n_plp": int(len(plp)), "counts": summary["n_variants"].to_dict(), "caveats": CAVEATS}, indent=2))

## Phase 4 — Safety and specificity

Essentiality (`isEssential` plus DepMap gene effect aggregated across cell lines), role in cancer, known safety liabilities, and baseline expression — all from Open Targets.

**Measuring enrichment properly.** The naive ratio of nervous expression to a gene's own highest tissue is misleading: a gene expressed equally everywhere scores perfectly. Enrichment here is the maximum nervous-tissue median divided by the gene's own median across tissues, with a pseudocount so that a gene silent in most tissues — exactly the brain-restricted case — is not divided by zero.

**Tissue selection is declared explicitly**, by biosample name read off the printed inventory, not by keyword guessing: an earlier keyword pass matched "cortex of kidney" as nervous tissue, which is the kind of error that is invisible unless the matched names are printed.

**Two resources, two questions.** Bulk GTEx asks whether a gene concentrates in nervous *tissue*; single-cell data asks whether it concentrates in neural *cell types*. Their disagreement is informative: *single-cell only* marks a sparse, scattered cell type that bulk dilutes away — the expected pattern for a sensory neuropathy — while *bulk only* raises the question of which cell type is responsible.

In [ ]:
# --- Phase 4 configuration + light safety annotation (Open Targets) ---------------------------------
if "universe" not in globals() or "class_source" not in universe.columns:
    universe = pd.read_csv(PROC / "02c_universe_curated.csv")
    _m1 = json.loads((PROC / "01_manifest.json").read_text())
    DISEASE_ID, DISEASE_NAME = _m1["disease_id"], _m1["disease_name"]
pool_ids = universe.loc[universe["in_pool"], "ensembl_id"].tolist()

DEPMAP_DEPENDENCY = -0.5     # gene effect below this = the cell line depends on the gene
DEPMAP_STRONG = -1.0         # conventional "as essential as a common-essential gene"
print(f"Phase 4 on the curated pool: {len(pool_ids)} genes")

SAFETY_Q = '''
query safety($ids: [String!]!) {
  targets(ensemblIds: $ids) {
    id
    isEssential
    hallmarks { cancerHallmarks { label impact } attributes { name } }
    safetyLiabilities { event datasource effects { direction dosing } }
  }
}'''
rows, all_attrs = [], set()
for i in range(0, len(pool_ids), 25):
    for t in ot_query(SAFETY_Q, {"ids": pool_ids[i:i + 25]})["targets"]:
        hm = t.get("hallmarks") or {}
        ch = hm.get("cancerHallmarks") or []
        attrs = sorted({a["name"] for a in (hm.get("attributes") or [])})
        all_attrs.update(attrs)
        liab = t.get("safetyLiabilities") or []
        rows.append({
            "ensembl_id": t["id"],
            "ot_is_essential": t.get("isEssential"),
            "n_cancer_hallmarks": len(ch),
            "hallmark_promotes": sum(1 for h in ch if (h.get("impact") or "").lower() == "promotes"),
            "hallmark_labels": "; ".join(sorted({h["label"] for h in ch})[:4]),
            "hallmark_attributes": "; ".join(attrs),          # kept raw, parsed below
            "n_safety_liabilities": len(liab),
            "safety_events": "; ".join(sorted({s["event"] for s in liab if s.get("event")})[:4]),
            "safety_directions": "; ".join(sorted({(ef.get("direction") or "") for s in liab
                                                   for ef in (s.get("effects") or []) if ef.get("direction")})),
        })
    print(f"  safety annotation {min(i + 25, len(pool_ids))}/{len(pool_ids)}")
safety = pd.DataFrame(rows)

# The cancer-role match is driven by what the API actually returns, so print the vocabulary once.
print("\nDistinct hallmark attribute names returned:", "; ".join(sorted(all_attrs)[:40]) or "none")
CANCER_ROLE_WORDS = ["oncogene", "suppressor", "fusion", "translocation", "amplification"]
safety["role_in_cancer"] = safety["hallmark_attributes"].fillna("").apply(
    lambda s: "; ".join(sorted({p.strip() for p in s.split(";")
                                if any(w in p.lower() for w in CANCER_ROLE_WORDS)})))
display(safety["ot_is_essential"].value_counts(dropna=False).to_frame("n"))
print("Genes with cancer hallmarks:", int((safety["n_cancer_hallmarks"] > 0).sum()),
      "· with a cancer-role attribute:", int((safety["role_in_cancer"] != "").sum()),
      "· with safety liabilities:", int((safety["n_safety_liabilities"] > 0).sum()))
print("Safety-liability directions seen:",
      "; ".join(sorted({d for s in safety["safety_directions"] for d in s.split("; ") if d})) or "none")

In [ ]:
# --- DepMap essentiality, one gene at a time, cached ---------------------------------------------
DEPMAP_Q = '''
query dep($id: String!) {
  target(ensemblId: $id) {
    depMapEssentiality { tissueName screens { depmapId geneEffect } }
  }
}'''
dep_cache_path = RAW / "depmap_cache.json"
dep_cache = json.loads(dep_cache_path.read_text()) if dep_cache_path.exists() else {}

todo = [g for g in pool_ids if g not in dep_cache]
print(f"Querying DepMap essentiality for {len(todo)} genes ({len(dep_cache)} cached)...")
for k, gid in enumerate(todo, 1):
    try:
        t = ot_query(DEPMAP_Q, {"id": gid})["target"] or {}
        eff = [s["geneEffect"] for blk in (t.get("depMapEssentiality") or [])
               for s in blk["screens"] if s.get("geneEffect") is not None]
        dep_cache[gid] = {"n_screens": len(eff),
                          "mean_gene_effect": float(np.mean(eff)) if eff else None,
                          "min_gene_effect": float(np.min(eff)) if eff else None,
                          "frac_dependent": float(np.mean([e < DEPMAP_DEPENDENCY for e in eff])) if eff else None,
                          "frac_strong": float(np.mean([e < DEPMAP_STRONG for e in eff])) if eff else None}
    except Exception as e:
        print(f"  {gid}: failed ({e})")
        dep_cache[gid] = {"n_screens": 0, "mean_gene_effect": None, "min_gene_effect": None,
                          "frac_dependent": None, "frac_strong": None}
    if k % 20 == 0:
        dep_cache_path.write_text(json.dumps(dep_cache)); print(f"  {k}/{len(todo)}")
dep_cache_path.write_text(json.dumps(dep_cache))

depmap = pd.DataFrame([{"ensembl_id": g, **dep_cache[g]} for g in pool_ids])
print("Genes with DepMap data:", int((depmap["n_screens"] > 0).sum()), "of", len(depmap))
display(depmap["mean_gene_effect"].describe().to_frame())

In [ ]:
# --- Baseline expression: fetch complete rows, then show what biosamples exist -----------------------
EXPR_Q = '''
query expr($id: String!, $size: Int!) {
  target(ensemblId: $id) {
    baselineExpression(page: {index: 0, size: $size}) {
      count
      rows { datasourceId datatypeId unit median specificity_score
             tissueBiosample { biosampleName } celltypeBiosample { biosampleName } }
    }
  }
}'''
EXPR_PAGE = 3000          # raised: the previous 500 silently cut GTEx down to ~17 of its ~54 tissues
expr_cache_path = RAW / f"expression_cache_{EXPR_PAGE}.json"
expr_cache = json.loads(expr_cache_path.read_text()) if expr_cache_path.exists() else {}

todo = [g for g in pool_ids if g not in expr_cache]
print(f"Querying baseline expression for {len(todo)} genes ({len(expr_cache)} cached)...")
for k, gid in enumerate(todo, 1):
    try:
        t = ot_query(EXPR_Q, {"id": gid, "size": EXPR_PAGE})["target"] or {}
        blk = t.get("baselineExpression") or {"count": 0, "rows": []}
        expr_cache[gid] = {"count": blk["count"], "rows": blk["rows"]}
    except Exception as err:
        print(f"  {gid}: failed -- {str(err)[:300]}")
        expr_cache[gid] = {"count": 0, "rows": []}
    if k % 20 == 0:
        expr_cache_path.write_text(json.dumps(expr_cache)); print(f"  {k}/{len(todo)}")
expr_cache_path.write_text(json.dumps(expr_cache))

expr_rows, truncated = [], []
for gid in pool_ids:
    blk = expr_cache.get(gid) or {"count": 0, "rows": []}
    if blk["count"] > len(blk["rows"]):
        truncated.append((gid, blk["count"]))
    for r in blk["rows"]:
        tis = (r.get("tissueBiosample") or {}).get("biosampleName")
        cel = (r.get("celltypeBiosample") or {}).get("biosampleName")
        expr_rows.append({"ensembl_id": gid, "datasource": r["datasourceId"], "datatype": r["datatypeId"],
                          "unit": r.get("unit"), "median": r.get("median"),
                          "specificity": r.get("specificity_score"), "tissue": tis, "celltype": cel,
                          "biosample": " / ".join(x for x in (tis, cel) if x)})
expr = pd.DataFrame(expr_rows)
if truncated:
    print(f"STILL TRUNCATED for {len(truncated)} genes (largest count {max(c for _, c in truncated)}): raise EXPR_PAGE")
else:
    print("No truncation: every gene's rows were returned in full.")

print("\nDatasource / datatype inventory:")
display(expr.groupby(["datasource", "datatype", "unit"])
            .agg(n_rows=("median", "size"), n_genes=("ensembl_id", "nunique"),
                 n_biosamples=("biosample", "nunique")).sort_values("n_genes", ascending=False))
for ds in expr["datasource"].unique():
    names = sorted(expr.loc[expr["datasource"] == ds, "biosample"].dropna().unique())
    print(f"\n--- {ds}: {len(names)} biosamples ---")
    print("; ".join(names))

In [ ]:
# --- Declared tissue selection and nervous-tissue enrichment ------------------------------------------
# Biosample names copied verbatim from the inventory printed by cell 38.
EXPR_SOURCE = "gtex"              # 50 bulk tissues, TPM, all 78 genes covered
EXPR_FLOOR = 1.0                  # below this TPM the gene is barely detected: ratios mean little

NERVOUS_BIOSAMPLES = [            # every central-nervous-system tissue GTEx provides here
    "Ammon's horn", "C1 segment of cervical spinal cord", "amygdala", "anterior cingulate cortex",
    "caudate nucleus", "cerebellum", "dorsolateral prefrontal cortex", "frontal cortex",
    "hypothalamus", "nucleus accumbens", "putamen", "substantia nigra",
]
SKIN_BIOSAMPLES = ["lower leg skin", "suprapubic skin"]      # anhidrosis affects skin and sweat glands
NERVOUS_FOLD_MIN = 2.0

# Second opinion at cell-type level. Microglia are deliberately excluded: they are macrophage-lineage,
# not neural. These are cell types, so they are matched on the celltype field, not the tissue field.
NEURAL_CELLTYPES = ["neuron", "retinal bipolar neuron", "Schwann cell", "enteroglial cell", "glial cell"]

e = expr[(expr["datasource"] == EXPR_SOURCE) & expr["median"].notna()].copy()
avail = set(e["biosample"].dropna())
missing = [b for b in NERVOUS_BIOSAMPLES + SKIN_BIOSAMPLES if b not in avail]
if missing:
    raise ValueError("Declared biosamples absent from the data: " + "; ".join(missing))
e["is_nervous"] = e["biosample"].isin(NERVOUS_BIOSAMPLES)
e["is_skin"] = e["biosample"].isin(SKIN_BIOSAMPLES)
print(f"{EXPR_SOURCE}: {e['ensembl_id'].nunique()} genes · {e['biosample'].nunique()} biosamples · "
      f"{len(NERVOUS_BIOSAMPLES)} declared nervous · {len(SKIN_BIOSAMPLES)} declared skin")

g = e.groupby("ensembl_id")
expr_sum = pd.DataFrame({
    "expr_max_overall": g["median"].max(),
    "expr_median_all": g["median"].median(),          # the gene's typical level across tissues
    "expr_max_nervous": e[e["is_nervous"]].groupby("ensembl_id")["median"].max(),
    "expr_max_skin": e[e["is_skin"]].groupby("ensembl_id")["median"].max(),
    "ot_specificity": g["specificity"].max(),
    "top_tissue": g.apply(lambda d: d.loc[d["median"].idxmax(), "biosample"], include_groups=False),
}).reset_index()
# Enrichment, not presence: a ubiquitous gene scores ~1 because its nervous level equals its own typical level.
expr_sum["nervous_fold"] = (expr_sum["expr_max_nervous"] / expr_sum["expr_median_all"].replace(0, np.nan)).round(2)
expr_sum["nervous_share_of_max"] = (expr_sum["expr_max_nervous"] / expr_sum["expr_max_overall"]).round(3)
expr_sum["low_expression"] = expr_sum["expr_max_overall"] < EXPR_FLOOR

# --- Second opinion: single-cell neural cell types ----------------------------------------------------
ts = expr[(expr["datasource"] == "tabula_sapiens") & expr["median"].notna()].copy()
if len(ts):
    ts["is_neural"] = ts["celltype"].isin(NEURAL_CELLTYPES)
    found = sorted(ts.loc[ts["is_neural"], "celltype"].unique())
    print("Neural cell types matched in tabula_sapiens:", "; ".join(found) or "none")
    tg = ts.groupby("ensembl_id")
    neural = pd.DataFrame({
        "ts_median_all": tg["median"].median(),
        "ts_max_neural": ts[ts["is_neural"]].groupby("ensembl_id")["median"].max(),
    }).reset_index()
    neural["neural_fold_sc"] = (neural["ts_max_neural"] / neural["ts_median_all"].replace(0, np.nan)).round(2)
    expr_sum = expr_sum.merge(neural[["ensembl_id", "neural_fold_sc"]], on="ensembl_id", how="left")
else:
    expr_sum["neural_fold_sc"] = np.nan

sym = universe.set_index("ensembl_id")["symbol"]
low = expr_sum.loc[expr_sum["low_expression"], "ensembl_id"]
print(f"\nBelow the {EXPR_FLOOR} TPM floor in every GTEx tissue (read their ratios with care):",
      ", ".join(sorted(sym.reindex(low).dropna())) or "none")
out = expr_sum.assign(symbol=expr_sum["ensembl_id"].map(sym)).sort_values("nervous_fold", ascending=False)
display(out[["symbol", "expr_max_overall", "expr_max_nervous", "nervous_fold", "nervous_share_of_max",
             "neural_fold_sc", "top_tissue", "low_expression"]].head(20))

In [ ]:
# --- Combine: safety / specificity table for the curated pool -------------------------------------
pool = (universe.loc[universe["in_pool"]]
          .merge(safety, on="ensembl_id", how="left")
          .merge(depmap, on="ensembl_id", how="left")
          .merge(expr_sum, on="ensembl_id", how="left"))

pool["flag_essential"] = (pool["ot_is_essential"].fillna(False)) | (pool["frac_strong"].fillna(0) > 0.5)
pool["flag_cancer_role"] = pool["role_in_cancer"].fillna("").ne("")
pool["flag_safety"] = pool["n_safety_liabilities"].fillna(0) > 0
pool["nervous_enriched"] = ((pool["nervous_fold"].fillna(0) >= NERVOUS_FOLD_MIN)
                            & ~pool["low_expression"].fillna(True))
pool["n_flags"] = pool[["flag_essential", "flag_cancer_role", "flag_safety"]].sum(axis=1)

TIER_ORDER = ["seed", "C · sister disease", "B · curated", "B · STRING only", "A · genetic/clinical"]
pool = (pool.assign(_t=lambda d: d["tier"].map({t: i for i, t in enumerate(TIER_ORDER)}))
            .sort_values(["_t", "n_flags", "symbol"]).drop(columns="_t"))
cols = ["symbol", "tier", "mech_class", "tdl", "chem_label", "ot_is_essential", "mean_gene_effect",
        "frac_dependent", "role_in_cancer", "n_safety_liabilities", "nervous_fold", "neural_fold_sc", "low_expression", "top_tissue", "n_flags"]
pool[cols].to_csv(PROC / "04_pool_safety_specificity.csv", index=False)

print("Flag counts across the curated pool:")
display(pool[["flag_essential", "flag_cancer_role", "flag_safety", "nervous_enriched"]].sum().to_frame("n"))
print("\nThe candidates proposed as neuron-enriched (hypothesis under test):")
WATCH = ["SHC3", "KIDINS220", "RIT2", "SH2B1", "SH2B2", "PRDM12", "ZFHX2", "NTF3", "NGF", "NTRK1", "NTRK2"]
display(pool.loc[pool["symbol"].isin(WATCH), cols])
print("\nUnflagged genes with nervous enrichment (safest, most specific):")
display(pool.loc[pool["nervous_enriched"] & pool["n_flags"].eq(0), cols].head(25))
print("\nBelow the detection floor in GTEx -- not judged on enrichment, shown so they are not lost:")
display(pool.loc[pool["low_expression"].fillna(False), cols])

In [ ]:
# --- Figure: safety map (specificity x essentiality) --------------------------------------------------
TIER_STYLE = {"C · sister disease": dict(color="#eb6834", filled=True),
              "B · curated": dict(color="#2a78d6", filled=True),
              "B · STRING only": dict(color="#2a78d6", filled=False),
              "A · genetic/clinical": dict(color="#1baf7a", filled=True)}
m = pool.dropna(subset=["nervous_fold", "mean_gene_effect"]).copy()
m = m[m["nervous_fold"] > 0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.axhline(DEPMAP_DEPENDENCY, color="#c3c2b7", lw=1, ls="--", zorder=0)
ax.axvline(NERVOUS_FOLD_MIN, color="#c3c2b7", lw=1, ls="--", zorder=0)
for t, st in TIER_STYLE.items():
    d = m[m["tier"] == t]
    if not len(d): continue
    ax.scatter(d["nervous_fold"], d["mean_gene_effect"], s=46, lw=1.4, zorder=3,
               facecolors=st["color"] if st["filled"] else "white", edgecolors=st["color"],
               label=f"{t} (n={len(d)})")
s = m[m["is_seed"].astype(bool)]
ax.scatter(s["nervous_fold"], s["mean_gene_effect"], marker="*", s=260, color="#0b0b0b", zorder=4,
           label="causal gene")
OFFSETS = [(6, 4), (6, -10), (-6, 7), (-6, -11)]        # cycle offsets so labels collide less
j = 0
for _, r in m.sort_values("nervous_fold").iterrows():
    if r["symbol"] in WATCH or r["is_seed"] or r["mean_gene_effect"] < DEPMAP_DEPENDENCY:
        dx, dy = OFFSETS[j % len(OFFSETS)]; j += 1
        ax.annotate(r["symbol"], (r["nervous_fold"], r["mean_gene_effect"]), xytext=(dx, dy),
                    textcoords="offset points", fontsize=8, color="#0b0b0b",
                    ha="left" if dx > 0 else "right")
ax.set_xscale("log")
ax.set_xlabel("nervous-tissue enrichment, log  (max nervous median ÷ the gene's own median across tissues)",
              color="#52514e")
ax.set_ylabel("DepMap mean gene effect\n(lower = cell lines depend on it)", color="#52514e")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.text(0.99, 0.02, "neuron-enriched and dispensable\n(safer, more specific)", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=8, color="#52514e", style="italic")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8)
ax.set_title(f"Safety map · {DISEASE_NAME} curated pool · expression source: {EXPR_SOURCE}", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "04_safety_map.png", dpi=200); plt.show()
print("Not plotted (missing expression or DepMap):",
      ", ".join(pool.loc[~pool["ensembl_id"].isin(m["ensembl_id"]), "symbol"]) or "none")

In [ ]:
# --- Save Phase 4 outputs ----------------------------------------------------------------------
pool.to_csv(PROC / "04_pool_annotated_full.csv", index=False)
(PROC / "04_manifest.json").write_text(json.dumps({
    "phase": "04_safety_and_specificity", "run_date": RUN_DATE, "n_pool": len(pool),
    "depmap_dependency_threshold": DEPMAP_DEPENDENCY, "depmap_strong_threshold": DEPMAP_STRONG,
    "expression_datasource": EXPR_SOURCE, "nervous_biosamples": NERVOUS_BIOSAMPLES,
    "skin_biosamples": SKIN_BIOSAMPLES, "neural_celltypes_sc": NEURAL_CELLTYPES,
    "nervous_fold_min": NERVOUS_FOLD_MIN, "expression_floor_tpm": EXPR_FLOOR,
    "n_flag_essential": int(pool["flag_essential"].sum()),
    "n_flag_cancer_role": int(pool["flag_cancer_role"].sum()),
    "n_nervous_enriched": int(pool["nervous_enriched"].sum()),
    "n_low_expression": int(pool["low_expression"].fillna(False).sum())}, indent=2))

## Phase 4b — Mouse knockout phenotypes

Expression says where a gene is. A knockout says what happens when it is gone. This is the only functional evidence available without an experiment, and it is orthogonal to everything above, so it cannot be inflated by the same artefacts.

Phenotype labels are graded into three declared tiers by closeness to this disease, with **explicit exclusions** for eye, ear, motor, enteric and muscle phenotypes — without them, "ganglion", "sensory" and "axon" match the entire nervous system, and a retinal phenotype scores as if it were nociceptive. The matched and excluded labels are both printed.

**Limit:** absence of a phenotype is not evidence of dispensability. It may mean no one made the mouse, or that the relevant assay was never run; genes with no model are kept visible rather than scored as negatives.

In [ ]:
# --- Phase 4b: mouse knockout phenotypes (orthogonal to expression) -----------------------------------
# Expression says where a gene is. This says what happens when it is removed.
MP_Q = '''
query mp($id: String!) {
  target(ensemblId: $id) {
    mousePhenotypes {
      targetInModel
      modelPhenotypeId
      modelPhenotypeLabel
      modelPhenotypeClasses { id label }
    }
  }
}'''
mp_cache_path = RAW / "mouse_phenotypes_cache.json"
mp_cache = json.loads(mp_cache_path.read_text()) if mp_cache_path.exists() else {}

todo = [g for g in pool_ids if g not in mp_cache]
print(f"Querying mouse phenotypes for {len(todo)} genes ({len(mp_cache)} cached)...")
for k, gid in enumerate(todo, 1):
    try:
        t = ot_query(MP_Q, {"id": gid})["target"] or {}
        mp_cache[gid] = t.get("mousePhenotypes") or []
    except Exception as err:
        print(f"  {gid}: failed -- {str(err)[:300]}")
        mp_cache[gid] = []
    if k % 20 == 0:
        mp_cache_path.write_text(json.dumps(mp_cache)); print(f"  {k}/{len(todo)}")
mp_cache_path.write_text(json.dumps(mp_cache))

mp_rows = []
for gid in pool_ids:
    for p in mp_cache.get(gid) or []:
        classes = [c["label"] for c in (p.get("modelPhenotypeClasses") or [])]
        mp_rows.append({"ensembl_id": gid, "mouse_gene": p.get("targetInModel"),
                        "phenotype_id": p.get("modelPhenotypeId"),
                        "phenotype": p.get("modelPhenotypeLabel") or "",
                        "classes": "; ".join(classes)})
mp = pd.DataFrame(mp_rows)
genes_with_model = mp["ensembl_id"].nunique() if len(mp) else 0
print(f"\n{genes_with_model} of {len(pool_ids)} genes have a mouse model · {len(mp)} phenotype records")
print("\nMost frequent top-level phenotype classes (the vocabulary to declare against):")
cls_counts = (mp["classes"].str.split("; ").explode().value_counts().head(25)
              if len(mp) else pd.Series(dtype=int))
display(cls_counts.to_frame("n_records"))

In [ ]:
# --- Declared phenotype tiers: how close is the knockout phenotype to THIS disease? -------------------
# The first pass matched any ganglion, any axon, any sensory structure, so retinal, cochlear, enteric and
# motor phenotypes scored as if they were nociceptive. Terms are now tiered, with explicit exclusions.
NOT_SOMATOSENSORY = ["cochlea", "vestibul", "inner ear", "retina", "optic", "olfactor", "taste",
                     "motor neuron", "enteric", "to muscle", "cardiac", "photoreceptor"]

PHENO_TIERS = {
    # Directly the phenotypes that define this disease
    "A · nociception, sensory ganglia, sweat glands": [
        "nocicept", "pain", "analgesi", "hyperalgesi", "allodyni",
        "dorsal root ganglion", "sensory ganglion", "sensory neuron", "sensory capabilities",
        "skin sensory", "sweat gland"],
    # Peripheral nervous structure and wiring: relevant, but not specific to pain
    "B · peripheral nerve, axons, autonomic": [
        "axon", "neurite", "innervation", "myelin", "peripheral nerve", "sympathetic",
        "cranial ganglia", "neuron number", "neuron degeneration", "thermoregulat", "core body temperature"],
    # Skin and appendages: the anhidrosis side of the phenotype, indirectly
    "C · skin and appendages": [
        "skin", "epiderm", "keratin", "hair follicle", "sebaceous"],
}
BROAD_NERVOUS_CLASSES = ["nervous system phenotype", "behavior/neurological phenotype"]
TIER_POINTS_PHENO = {"A · nociception, sensory ganglia, sweat glands": 3,
                     "B · peripheral nerve, axons, autonomic": 2,
                     "C · skin and appendages": 2}

if len(mp):
    low = mp["phenotype"].str.lower()
    excluded = low.apply(lambda s: any(w in s for w in NOT_SOMATOSENSORY))
    for tier, words in PHENO_TIERS.items():
        hit = low.apply(lambda s: any(w in s for w in words))
        # Exclusions apply to the two neural tiers; a skin phenotype is not an eye phenotype.
        mp[tier] = hit & (~excluded if tier.startswith(("A", "B")) else True)
    mp["broad_nervous"] = mp["classes"].apply(lambda s: any(c in s.lower() for c in BROAD_NERVOUS_CLASSES))

    print(f"Phenotype records excluded as non-somatosensory (eye, ear, motor, enteric): {int(excluded.sum())}")
    for tier in PHENO_TIERS:
        labels = sorted(mp.loc[mp[tier], "phenotype"].unique())
        print(f"\n--- {tier}: {len(labels)} distinct phenotypes ---")
        print("; ".join(labels[:20]) + (" ..." if len(labels) > 20 else "") if labels else "none")
    dropped = sorted(mp.loc[excluded & (low.str.contains("ganglion|sensory|axon|innervation")), "phenotype"].unique())
    print(f"\nExcluded despite looking neural ({len(dropped)}):")
    print("; ".join(dropped[:20]) + (" ..." if len(dropped) > 20 else ""))

    agg = {t: (t, "any") for t in PHENO_TIERS}
    agg["broad_nervous"] = ("broad_nervous", "any")
    agg["n_phenotypes"] = ("phenotype", "size")
    agg["mouse_gene"] = ("mouse_gene", "first")
    mouse = mp.groupby("ensembl_id").agg(**agg).reset_index()
    mouse["disease_relevant_pheno"] = mouse[list(PHENO_TIERS)].any(axis=1)
    def summarise(gid):
        d = mp[mp["ensembl_id"] == gid]
        hits = sorted({p for t in PHENO_TIERS for p in d.loc[d[t], "phenotype"]})
        return "; ".join(hits[:6])
    mouse["relevant_phenotypes"] = mouse["ensembl_id"].map(summarise)
else:
    mouse = pd.DataFrame(columns=["ensembl_id", *PHENO_TIERS, "broad_nervous", "n_phenotypes",
                                  "mouse_gene", "disease_relevant_pheno", "relevant_phenotypes"])

pool = pool.drop(columns=[c for c in mouse.columns if c != "ensembl_id" and c in pool.columns]) \
           .merge(mouse, on="ensembl_id", how="left")
for b in [*PHENO_TIERS, "broad_nervous", "disease_relevant_pheno"]:
    pool[b] = pool[b].eq(True)                      # eq(True) avoids the fillna downcasting warning
pool["has_mouse_model"] = pool["n_phenotypes"].notna()

print("\nGenes per phenotype tier (curated pool):")
display(pool[[*PHENO_TIERS, "broad_nervous", "disease_relevant_pheno", "has_mouse_model"]].sum().to_frame("n_genes"))
print("\nTier A genes -- the phenotypes that define this disease:")
display(pool.loc[pool[list(PHENO_TIERS)[0]], ["symbol", "tier", "mech_class", "mouse_gene", "relevant_phenotypes"]])
pool.to_csv(PROC / "04b_pool_with_mouse.csv", index=False)
(PROC / "04b_manifest.json").write_text(json.dumps({
    "phase": "04b_mouse_phenotypes", "run_date": RUN_DATE, "pheno_tiers": PHENO_TIERS,
    "not_somatosensory": NOT_SOMATOSENSORY, "tier_points": TIER_POINTS_PHENO,
    "broad_nervous_classes": BROAD_NERVOUS_CLASSES, "n_with_model": int(pool["has_mouse_model"].sum()),
    "n_tier_a": int(pool[list(PHENO_TIERS)[0]].sum())}, indent=2))

## Phase 5 — Transparent scoring, strategy, evidence cards

Four declared axes, scored separately and **never averaged into one another**: disease link (3) + tissue specificity (2) + safety (2) + mouse knockout (3). Being understudied is reported as a marker, not as points — it is an opportunity, not evidence.

Each candidate carries the strategy its mechanism implies, the **direction** the target would have to move (almost always *activate*, in a loss-of-function disease), and which variant groups from Phase 3 that strategy could serve. Targets whose existing drugs are inhibitors are flagged: chemical matter pointing the wrong way is not an asset.

**What the score does not measure.** It ranks target quality, not strategy plausibility. A target can score perfectly while the strategy it implies presumes cells this disease has already lost. That is why strategy and genotype coverage stay in columns of their own, beside the total rather than inside it.

In [ ]:
# --- Phase 5, step 1: four declared axes, scored separately and never averaged together ---------------
NEURAL_FOLD_MIN = 2.0      # threshold on the single-cell measure
if "NERVOUS_FOLD_MIN" not in globals():   # normally declared in cell 38b; kept in step with it
    NERVOUS_FOLD_MIN = 2.0

# Axis 1 - how firmly the gene is tied to the disease
TIER_POINTS = {"seed": 3, "C · sister disease": 3, "B · curated": 2,
               "B · STRING only": 1, "A · genetic/clinical": 1}
pool["ax1_disease_link"] = pool["tier"].map(TIER_POINTS).fillna(0).astype(int)

# Axis 2 - tissue specificity, read as the agreement between two resources that ask different questions
bulk_hi = pool["nervous_fold"].fillna(0) >= NERVOUS_FOLD_MIN
sc_hi = pool["neural_fold_sc"].fillna(0) >= NEURAL_FOLD_MIN
pool["specificity_case"] = np.select(
    [bulk_hi & sc_hi, ~bulk_hi & sc_hi, bulk_hi & ~sc_hi],
    ["both", "single-cell only (sparse cell type)", "bulk only (which cell type?)"],
    default="neither")
pool.loc[pool["low_expression"].fillna(False), "specificity_case"] = "not assessable (below detection)"
pool["ax2_specificity"] = pool["specificity_case"].map(
    {"both": 2, "single-cell only (sparse cell type)": 2, "bulk only (which cell type?)": 1,
     "neither": 0, "not assessable (below detection)": 0}).astype(int)

# Axis 3 - safety: full marks only with no liability of any kind
pool["ax3_safety"] = np.select([pool["n_flags"] == 0, pool["n_flags"] == 1], [2, 1], default=0)

# Axis 4 - functional evidence from mouse knockouts, graded by how close the phenotype is to this disease
TIER_A, TIER_B, TIER_C = list(PHENO_TIERS)
pool["ax4_mouse"] = np.select(
    [pool[TIER_A], pool[TIER_B] | pool[TIER_C], pool["broad_nervous"], pool["has_mouse_model"]],
    [3, 2, 1, 0], default=0)
# A gene with no mouse model is unknown, not negative: kept visible rather than scored as a zero.
pool["mouse_evidence"] = np.select(
    [~pool["has_mouse_model"], pool[TIER_A], pool[TIER_B] | pool[TIER_C], pool["broad_nervous"]],
    ["no mouse model (unknown, not negative)", "disease-defining phenotype",
     "peripheral nerve / skin phenotype", "nervous or behavioural phenotype only"],
    default="model exists, nothing relevant")

AXES = ["ax1_disease_link", "ax2_specificity", "ax3_safety", "ax4_mouse"]
pool["score_total"] = pool[AXES].sum(axis=1)
# Not part of the score: an orthogonal marker of how unexplored the target is.
pool["opportunity_tdark"] = pool["tdl"].isin(["Tdark", "Tbio"]) & (pool["chem_level"] <= 1)

print("Score components (max 3 + 2 + 2 + 3 = 10). Each axis is reported, the total never hides them.")
display(pool["specificity_case"].value_counts().to_frame("n_genes"))
rank_cols = ["symbol", "tier", "mech_class", *AXES, "score_total", "specificity_case",
             "opportunity_tdark", "tdl", "chem_label"]
ranked = pool.sort_values(["score_total", "ax1_disease_link"], ascending=False).reset_index(drop=True)
display(ranked[rank_cols].head(25))

In [ ]:
# --- Phase 5, step 2: therapeutic strategy and which genotypes it could serve -------------------------
STRATEGY_BY_CLASS = {
    "causal gene":                                ("restore NTRK1 itself", "restore / replace"),
    "neurotrophin ligand / co-receptor":          ("raise signalling through TrkA", "activate"),
    "parallel receptor":                          ("recruit a parallel Trk receptor", "activate"),
    "GPCR transactivation route":                 ("transactivate TrkA without neurotrophin", "activate"),
    "docking adaptor":                            ("re-engage the pathway below the receptor", "activate"),
    "signalling effector":                        ("re-engage the pathway below the receptor", "activate"),
    "trafficking / signalling endosome":          ("restore retrograde signalling transport", "activate"),
    "transcriptional / developmental regulator":  ("redirect nociceptor specification", "activate"),
    "ion channel":                                ("restore excitability of surviving neurons", "activate"),
    "symptomatic (autonomic)":                    ("manage autonomic symptoms", "varies"),
}
# Which allele groups each strategy could serve (from the Phase 3 reasoning; 'possible' needs experiments)
STRATEGY_GENOTYPES = {
    "restore NTRK1 itself":                       "all genotypes",
    "raise signalling through TrkA":              "only alleles leaving kinase activity (~4% of P/LP variants)",
    "recruit a parallel Trk receptor":            "all genotypes (does not use TrkA)",
    "transactivate TrkA without neurotrophin":    "extracellular missense only, if the protein folds and traffics",
    "re-engage the pathway below the receptor":   "all genotypes (does not use TrkA)",
    "restore retrograde signalling transport":    "all genotypes (does not use TrkA)",
    "redirect nociceptor specification":          "all genotypes; developmental window applies",
    "restore excitability of surviving neurons":  "all genotypes, but only where neurons survived",
    "manage autonomic symptoms":                  "all genotypes; symptomatic only",
}
# A proposed next experiment per strategy. Proposed only: nothing here was performed.
VALIDATION_BY_STRATEGY = {
 "restore NTRK1 itself": "transduce patient-derived iPSC nociceptors with a wild-type NTRK1 cassette; read out TrkA phosphorylation and neurite outgrowth against an isogenic control",
 "raise signalling through TrkA": "apply the ligand to patient iPSC nociceptors carrying an extracellular missense allele; null-allele lines serve as the built-in negative control",
 "recruit a parallel Trk receptor": "check the parallel receptor is expressed in patient nociceptors, then test whether its agonist rescues neurite outgrowth with TrkA absent",
 "transactivate TrkA without neurotrophin": "first show the mutant receptor reaches the membrane, then ask whether a selective agonist phosphorylates TrkA without NGF",
 "re-engage the pathway below the receptor": "modulate the node in patient nociceptors and read out MAPK and PI3K activation plus neurite outgrowth; TrkA-null lines are the stringent test",
 "restore retrograde signalling transport": "compartmentalised microfluidic cultures; read out retrograde transport of labelled signalling endosomes and cell-body survival",
 "redirect nociceptor specification": "overexpress during iPSC differentiation and quantify nociceptor markers against an isogenic control",
 "restore excitability of surviving neurons": "patch-clamp or calcium imaging on patient iPSC nociceptors; test whether an activator restores firing",
 "manage autonomic symptoms": "not a mechanism study: assess thermoregulation and autonomic readouts in an established animal model",
}
# Per-gene strategy overrides: the mechanism class is too coarse for the ligands.
STRATEGY_OVERRIDE = {
    "GDNF":  ("recruit a parallel neurotrophic pathway (RET/GFRa)", "activate"),
    "BDNF":  ("recruit a parallel Trk receptor", "activate"),
    "NTF4":  ("recruit a parallel Trk receptor", "activate"),
    "NGFR":  ("modulate the p75 co-receptor", "unclear"),
    "SORT1": ("modulate neurotrophin sorting", "unclear"),
}
EXTRA_GENOTYPES = {
    "recruit a parallel neurotrophic pathway (RET/GFRa)": "all genotypes (does not use TrkA)",
    "modulate the p75 co-receptor": "uncertain: p75 both assists and opposes TrkA signalling",
    "modulate neurotrophin sorting": "all genotypes; affects how neurotrophins are routed",
}
EXTRA_VALIDATION = {
    "recruit a parallel neurotrophic pathway (RET/GFRa)":
        "check the receptor complex is present on patient nociceptors, then test whether the ligand rescues survival and neurite outgrowth with TrkA absent",
    "modulate the p75 co-receptor":
        "establish the direction first: compare TrkA-pathway output on patient nociceptors with p75 raised and lowered, before proposing either",
    "modulate neurotrophin sorting":
        "measure mature versus pro-neurotrophin balance in patient nociceptors and ask whether shifting it changes TrkA-pathway output",
}

def _strategy(r):
    return STRATEGY_OVERRIDE.get(r["symbol"], STRATEGY_BY_CLASS.get(r["mech_class"], ("unassigned", "unclear")))
pool["strategy"] = pool.apply(lambda r: _strategy(r)[0], axis=1)
pool["direction_needed"] = pool.apply(lambda r: _strategy(r)[1], axis=1)
STRATEGY_GENOTYPES.update(EXTRA_GENOTYPES); VALIDATION_BY_STRATEGY.update(EXTRA_VALIDATION)
pool["genotypes_served"] = pool["strategy"].map(STRATEGY_GENOTYPES).fillna("")
pool["proposed_validation"] = pool["strategy"].map(VALIDATION_BY_STRATEGY).fillna(
    "define a readout for this mechanism before proposing an experiment")
# Existing drugs are mostly inhibitors, so chemical matter is not automatically usable here.
pool["drugs_wrong_direction"] = (pool["n_drugs_clinical"].fillna(0) > 0) & pool["direction_needed"].eq("activate")

print("Candidates by strategy:")
display(pool.groupby(["strategy", "direction_needed"])
            .agg(n_genes=("symbol", "size"), mean_score=("score_total", "mean")).round(1)
            .sort_values("n_genes", ascending=False))
print("\nTargets whose existing drugs point the wrong way (inhibitors where activation is needed):")
display(pool.loc[pool["drugs_wrong_direction"], ["symbol", "strategy", "n_drugs_clinical", "chem_label"]]
            .sort_values("n_drugs_clinical", ascending=False).head(12))

In [ ]:
# --- Phase 5, step 3: the shortlist and its figure ----------------------------------------------------
TOP_N = 15
short = (pool.sort_values(["score_total", "ax1_disease_link", "nervous_fold"], ascending=False)
             .head(TOP_N).reset_index(drop=True))
short_cols = ["symbol", "tier", "mech_class", "strategy", "direction_needed", "score_total",
              *AXES, "specificity_case", "relevant_phenotypes", "opportunity_tdark"]
short[short_cols].to_csv(PROC / "05_shortlist.csv", index=False)
pool.sort_values("score_total", ascending=False).to_csv(PROC / "05_pool_scored.csv", index=False)
display(short[short_cols])

AX_COL = {"ax1_disease_link": "#2a78d6", "ax2_specificity": "#eb6834",
          "ax3_safety": "#1baf7a", "ax4_mouse": "#eda100"}
AX_NAME = {"ax1_disease_link": "disease link", "ax2_specificity": "tissue specificity",
           "ax3_safety": "safety", "ax4_mouse": "mouse knockout"}
d = short.iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 0.42 * len(d) + 2.2))
left = np.zeros(len(d))
for a, c in AX_COL.items():
    ax.barh(range(len(d)), d[a], left=left, color=c, height=0.62, edgecolor="white", linewidth=2,
            label=AX_NAME[a])
    for i, (v, l) in enumerate(zip(d[a], left)):
        if v > 0:
            ax.text(l + v / 2, i, int(v), ha="center", va="center", fontsize=8,
                    color="white" if a != "ax4_mouse" else "#1F2937")
    left += d[a].to_numpy()
labels = [f"{s}  ({t.split(' · ')[0]})" for s, t in zip(d["symbol"], d["tier"])]
ax.set_yticks(range(len(d)), labels, fontsize=9)
for i, star in enumerate(d["opportunity_tdark"]):
    if star:
        ax.text(left[i] + 0.12, i, "understudied", va="center", fontsize=7.5, color="#52514e", style="italic")
ax.set_xlim(0, 11)
ax.set_xlabel("declared score (each axis shown separately; the total never hides its parts)", color="#52514e")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4, frameon=False, fontsize=8.5)
ax.set_title(f"Prioritised candidates · {DISEASE_NAME}", fontsize=10, pad=24)
fig.tight_layout(); fig.savefig(FIG / "05_shortlist_scores.png", dpi=200); plt.show()

In [ ]:
# --- Phase 5, step 4: evidence cards + save -----------------------------------------------------------
def card(r):
    fold = lambda v: "n/a" if pd.isna(v) else f"{v:g}"
    lines = [
        f"### {r['symbol']} — {r['mech_class']}",
        f"**Score {int(r['score_total'])}/10** "
        f"(disease link {int(r['ax1_disease_link'])}/3 · specificity {int(r['ax2_specificity'])}/2 · "
        f"safety {int(r['ax3_safety'])}/2 · mouse {int(r['ax4_mouse'])}/3)",
        "",
        f"- **Why it is here:** {r['tier']}"
        + (f" — causal in {r['c_diseases']}" if isinstance(r.get("c_diseases"), str) and r["c_diseases"] else "")
        + (f" — interacts with the causal gene ({r['b_sources']})"
           if isinstance(r.get("b_sources"), str) and r["b_sources"] else ""),
        f"- **Proposed strategy:** {r['strategy']} — needs to **{r['direction_needed']}** the target",
        f"- **Genotypes it could serve:** {r['genotypes_served']}",
        f"- **Tissue specificity:** {r['specificity_case']} "
        f"(GTEx {fold(r['nervous_fold'])}×, single-cell {fold(r['neural_fold_sc'])}×; top GTEx tissue: {r['top_tissue']})",
        f"- **Mouse knockout:** " + (f"{r['relevant_phenotypes']}" if r["relevant_phenotypes"]
                                     else ("nervous/behavioural phenotype only" if r["broad_nervous"]
                                           else ("model exists, no relevant phenotype" if r["has_mouse_model"]
                                                 else "no mouse model in Open Targets"))),
        f"- **Safety:** DepMap mean gene effect {r['mean_gene_effect']:.2f}"
        + (f"; cancer role: {r['role_in_cancer']}" if r["role_in_cancer"] else "")
        + (f"; {int(r['n_safety_liabilities'])} known safety liabilities" if r["n_safety_liabilities"] else ""),
        f"- **Development level:** {r['tdl']} · chemical matter: {r['chem_label']}"
        + (" — **existing drugs are inhibitors, the opposite of what is needed**"
           if r["drugs_wrong_direction"] else ""),
        f"- **Proposed validation (not performed):** {r['proposed_validation']}.",
        "",
    ]
    return "\n".join(lines)

header = [f"# Candidate evidence cards — {DISEASE_NAME}", "",
          f"Generated {RUN_DATE} from Open Targets, Pharos/IDG, ClinVar, UniProt, AlphaFold DB.",
          "Computational prioritisation only: no experimental validation was performed.", ""]
cards = header + [card(r) for _, r in short.iterrows()]
card_path = PROC / "05_evidence_cards.md"
card_path.write_text("\n".join(cards))
print("\n".join(cards[:len(header) + 2]))
print(f"... {len(short)} cards written to {card_path.name}")

(PROC / "05_manifest.json").write_text(json.dumps({
    "phase": "05_scoring_and_cards", "run_date": RUN_DATE, "tier_points": TIER_POINTS,
    "nervous_fold_min": NERVOUS_FOLD_MIN, "neural_fold_min": NEURAL_FOLD_MIN,
    "axes": {a: AX_NAME[a] for a in AXES}, "max_score": 10, "top_n": TOP_N,
    "strategy_by_class": {k: list(v) for k, v in STRATEGY_BY_CLASS.items()},
    "strategy_genotypes": STRATEGY_GENOTYPES,
    "validation_by_strategy": VALIDATION_BY_STRATEGY}, indent=2))

## Outputs

Every phase writes a dated snapshot of its API responses, one or more tables, and a manifest recording every declared threshold, keyword list and manual decision. The cell below archives all of it.

**What is not in here:** any experimental result. The proposed validations in the evidence cards are proposals.

In [ ]:
# --- Archive everything for download -------------------------------------------------------------------
manifests = sorted(PROC.glob("*_manifest.json"))
print("Phases completed:", ", ".join(json.loads(m.read_text())["phase"] for m in manifests))
print("Tables:", ", ".join(sorted(p.name for p in PROC.glob("*.csv"))))
print("Figures:", ", ".join(sorted(p.name for p in FIG.glob("*.png"))))
archive = shutil.make_archive(f"/content/rare-target-prioritization_{RUN_DATE}", "zip", ROOT)
from google.colab import files
files.download(archive)